# 06 — LLM Batch Inference

## Purpose

This notebook executes the locked paired LLM experiment for the primary 46-feature condition.

For each of the 200 evaluation records, the same underlying feature–value payload is submitted in two information-equivalent representations:

1. structured JSON; and
2. deterministic natural-language text.

The notebook first resolves the provider-specific completion constraint identified in Notebook 05. MiniMax-M3 exhausted a 4,096-token completion allowance under its default thinking behaviour, while the 8,192-token retry exceeded the previous 60-second client timeout.

Before batch inference, this notebook therefore tests whether the UoA Agentic Gateway accepts explicit disabling of MiniMax-M3 thinking. No batch request is permitted until the paired preflight gate returns complete, schema-valid responses for both representation conditions.

Ground-truth labels remain private and are not loaded or transmitted during inference.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from openai import OpenAI


def find_project_root(start: Path) -> Path:
    """Locate the repository root from the project or notebooks directory."""
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "notebooks").is_dir()
            and (candidate / "configs").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing "
        "README.md, notebooks/ and configs/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

REPRESENTATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "representations"
    / "primary_46"
)

STRUCTURED_PATH = REPRESENTATION_DIR / "structured.jsonl"
TEXT_PATH = REPRESENTATION_DIR / "deterministic_text.jsonl"
EQUIVALENCE_PATH = REPRESENTATION_DIR / "equivalence_validation.csv"
REPRESENTATION_MANIFEST_PATH = REPRESENTATION_DIR / "manifest.json"

PROTOCOL_PATH = (
    PROJECT_ROOT / "configs" / "llm_protocol_primary_46.json"
)
OUTPUT_SCHEMA_PATH = (
    PROJECT_ROOT / "configs" / "llm_output_schema.json"
)

required_paths = [
    STRUCTURED_PATH,
    TEXT_PATH,
    EQUIVALENCE_PATH,
    REPRESENTATION_MANIFEST_PATH,
    PROTOCOL_PATH,
    OUTPUT_SCHEMA_PATH,
]

missing_paths = [
    path.relative_to(PROJECT_ROOT).as_posix()
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required files are missing:\n- " + "\n- ".join(missing_paths)
    )

pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "representation_directory": str(REPRESENTATION_DIR),
        "required_files_found": len(required_paths),
        "missing_required_files": len(missing_paths),
        "api_key_present": bool(os.getenv("UOA_API_KEY")),
        "network_request_made": False,
    },
    name="value",
)

project_root                    /Users/ruiwang/Developer/compsci742-rui-pilot
representation_directory    /Users/ruiwang/Developer/compsci742-rui-pilot/...
required_files_found                                                        6
missing_required_files                                                      0
api_key_present                                                          True
network_request_made                                                    False
Name: value, dtype: object

In [2]:
def read_jsonl(path: Path) -> list[dict]:
    """Read a UTF-8 JSON Lines file."""
    records = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()

            if not stripped:
                continue

            try:
                records.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path.name}, line {line_number}."
                ) from exc

    return records


structured_records = read_jsonl(STRUCTURED_PATH)
text_records = read_jsonl(TEXT_PATH)
equivalence_validation = pd.read_csv(EQUIVALENCE_PATH)

with REPRESENTATION_MANIFEST_PATH.open(
    "r", encoding="utf-8"
) as handle:
    representation_manifest = json.load(handle)

with PROTOCOL_PATH.open("r", encoding="utf-8") as handle:
    llm_protocol = json.load(handle)

with OUTPUT_SCHEMA_PATH.open("r", encoding="utf-8") as handle:
    llm_output_schema = json.load(handle)


structured_sample_ids = [
    record["sample_id"] for record in structured_records
]
text_sample_ids = [
    record["sample_id"] for record in text_records
]

paired_hashes_match = all(
    structured_record["canonical_payload_sha256"]
    == text_record["canonical_payload_sha256"]
    for structured_record, text_record in zip(
        structured_records,
        text_records,
        strict=True,
    )
)

assert len(structured_records) == 200
assert len(text_records) == 200
assert len(equivalence_validation) == 200
assert structured_sample_ids == text_sample_ids
assert len(set(structured_sample_ids)) == 200
assert paired_hashes_match
assert llm_protocol["feature_set"]["feature_count"] == 46
assert llm_protocol["evaluation_inputs"]["record_count"] == 200
assert llm_output_schema["strict"] is True

input_validation_summary = pd.Series(
    {
        "structured_records": len(structured_records),
        "deterministic_text_records": len(text_records),
        "equivalence_validation_rows": len(equivalence_validation),
        "sample_order_matches": (
            structured_sample_ids == text_sample_ids
        ),
        "unique_sample_ids": len(set(structured_sample_ids)),
        "paired_payload_hashes_match": paired_hashes_match,
        "feature_count": llm_protocol["feature_set"]["feature_count"],
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

input_validation_summary

structured_records               200
deterministic_text_records       200
equivalence_validation_rows      200
sample_order_matches            True
unique_sample_ids                200
paired_payload_hashes_match     True
feature_count                     46
ground_truth_loaded            False
network_request_made           False
Name: value, dtype: object

## 1. Provider-specific constraint resolution

Notebook 05 established that the UoA Agentic Gateway accepts the selected model, `temperature=0.0`, `seed=742` and strict JSON-Schema output.

However, the paired research smoke test identified a provider-specific completion constraint:

- with a 4,096-token completion allowance, both representation conditions ended with `finish_reason="length"` before returning a complete visible answer;
- increasing the allowance to 8,192 tokens did not establish success because both requests reached the client's fixed 60-second timeout;
- the returned metadata indicated that MiniMax-M3 was producing substantial provider-specific reasoning content.

According to the MiniMax-M3 OpenAI-compatible API specification, thinking is enabled by default but can be explicitly disabled with:

`{"thinking": {"type": "disabled"}}`

This experiment requires a short forced binary decision and a schema-constrained list of five cited features. Unbounded internal reasoning is not itself an evaluated output and would substantially increase latency, token consumption and failure risk across 400 planned requests.

The following capability probe therefore tests whether the UoA gateway accepts explicit thinking control. The test:

- contains no research record or ground-truth label;
- makes at most one external request;
- disables automatic SDK retries;
- requests a minimal schema-constrained response;
- records whether any provider-specific reasoning content is still returned; and
- does not yet alter the locked provider-independent experimental protocol.

If supported, the same disabled-thinking setting will subsequently be applied to both input-representation conditions.

In [3]:
# The endpoint and model identifiers are those empirically verified in
# Notebook 05. They are repeated explicitly here so that this notebook can be
# executed independently without relying on the previous kernel state.
UOA_GATEWAY_BASE_URL = "https://agent.elliottwen.info/v1"
UOA_GATEWAY_MODEL = "MiniMax-M3"

# The previous 60-second client timeout prevented the 8,192-token diagnostic
# request from completing. A 180-second upper bound is used for capability
# resolution and later preflight testing. This remains a finite timeout and
# therefore cannot leave the notebook waiting indefinitely.
UOA_CLIENT_TIMEOUT_SECONDS = 180.0

# Automatic SDK retries are disabled. Each explicit function call therefore
# corresponds to at most one external request, preventing hidden duplicate
# requests and preserving an auditable request count.
UOA_AUTOMATIC_RETRIES = 0

# Use the same deterministic inference settings previously accepted by the
# gateway. These settings will remain identical across representation
# conditions.
INFERENCE_TEMPERATURE = 0.0
INFERENCE_SEED = 742

if not os.getenv("UOA_API_KEY"):
    raise EnvironmentError(
        "UOA_API_KEY is not available in the current Jupyter kernel. "
        "Set it in the environment without printing or storing the key."
    )

uoa_gateway_client = OpenAI(
    api_key=os.environ["UOA_API_KEY"],
    base_url=UOA_GATEWAY_BASE_URL,
    timeout=UOA_CLIENT_TIMEOUT_SECONDS,
    max_retries=UOA_AUTOMATIC_RETRIES,
)

client_configuration = pd.Series(
    {
        "base_url": UOA_GATEWAY_BASE_URL,
        "requested_model": UOA_GATEWAY_MODEL,
        "client_timeout_seconds": UOA_CLIENT_TIMEOUT_SECONDS,
        "automatic_retries": UOA_AUTOMATIC_RETRIES,
        "temperature": INFERENCE_TEMPERATURE,
        "seed": INFERENCE_SEED,
        "api_key_present": True,
        "api_key_value_recorded": False,
        "network_request_made": False,
    },
    name="value",
)

client_configuration

base_url                  https://agent.elliottwen.info/v1
requested_model                                 MiniMax-M3
client_timeout_seconds                               180.0
automatic_retries                                        0
temperature                                            0.0
seed                                                   742
api_key_present                                       True
api_key_value_recorded                               False
network_request_made                                 False
Name: value, dtype: object

### 1.1 Disabled-thinking capability probe

This cell makes exactly one quota-consuming request.

The probe uses a minimal response schema rather than a research record. Relative to the already validated gateway settings, the material provider-specific change is the addition of explicit disabled-thinking control.

The probe is considered successful only when:

1. the request completes without an API error;
2. the visible response is non-empty;
3. the response finishes normally rather than because of the token limit;
4. the visible response parses as JSON;
5. the parsed object contains exactly `{"status": "ok", "code": 742}`; and
6. no provider-specific reasoning content is returned.

Failure at this stage does not trigger an automatic retry and does not permit paired or batch inference.

In [4]:
# A deliberately small schema isolates provider capability from the research
# task. No sample, feature, label or dataset information is transmitted.
THINKING_CONTROL_PROBE_SCHEMA = {
    "name": "thinking_control_probe",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "status": {
                "type": "string",
                "enum": ["ok"],
            },
            "code": {
                "type": "integer",
                "enum": [742],
            },
        },
        "required": [
            "status",
            "code",
        ],
        "additionalProperties": False,
    },
}

# The expected visible object is retained locally for exact validation.
EXPECTED_THINKING_CONTROL_PROBE_OBJECT = {
    "status": "ok",
    "code": 742,
}

# With thinking disabled, 128 completion tokens should be substantially more
# than required for the small expected JSON object.
THINKING_CONTROL_PROBE_MAX_TOKENS = 128

# Prevent accidental duplicate execution in the same kernel. To perform a
# deliberate repeat, the existing result must first be inspected, documented
# and explicitly deleted.
if "thinking_control_probe_result" in globals():
    raise RuntimeError(
        "The disabled-thinking capability probe has already been executed "
        "in this kernel. Do not repeat a quota-consuming request accidentally."
    )

probe_started_at_utc = datetime.now(timezone.utc).isoformat()
probe_start_time = time.perf_counter()

try:
    thinking_control_response = (
        uoa_gateway_client.chat.completions.create(
            model=UOA_GATEWAY_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Return an object with status 'ok' and code 742. "
                        "Return only the schema-conforming object."
                    ),
                }
            ],
            temperature=INFERENCE_TEMPERATURE,
            seed=INFERENCE_SEED,
            max_tokens=THINKING_CONTROL_PROBE_MAX_TOKENS,
            response_format={
                "type": "json_schema",
                "json_schema": THINKING_CONTROL_PROBE_SCHEMA,
            },
            # MiniMax-M3 enables thinking by default on its OpenAI-compatible
            # Chat Completions API. This is the capability being tested.
            extra_body={
                "thinking": {
                    "type": "disabled",
                }
            },
        )
    )

    probe_elapsed_seconds = time.perf_counter() - probe_start_time
    probe_choice = thinking_control_response.choices[0]
    probe_message = probe_choice.message

    # Only the final visible response is required for the experiment.
    visible_response = (probe_message.content or "").strip()

    # Do not retain or display provider reasoning text. Its presence is
    # represented only by a Boolean and character count.
    reasoning_content = getattr(
        probe_message,
        "reasoning_content",
        None,
    )
    reasoning_character_count = len(reasoning_content or "")

    try:
        parsed_visible_response = json.loads(visible_response)
        visible_json_parsed = True
    except json.JSONDecodeError:
        parsed_visible_response = None
        visible_json_parsed = False

    usage = thinking_control_response.usage

    thinking_control_probe_result = {
        "probe_started_at_utc": probe_started_at_utc,
        "request_succeeded": True,
        "requested_model": UOA_GATEWAY_MODEL,
        "reported_model": getattr(
            thinking_control_response,
            "model",
            None,
        ),
        "system_fingerprint": getattr(
            thinking_control_response,
            "system_fingerprint",
            None,
        ),
        "finish_reason": probe_choice.finish_reason,
        "elapsed_seconds": probe_elapsed_seconds,
        "thinking_requested": "disabled",
        "visible_response": visible_response,
        "visible_response_character_count": len(visible_response),
        "visible_json_parsed": visible_json_parsed,
        "parsed_object_matches_expected": (
            parsed_visible_response
            == EXPECTED_THINKING_CONTROL_PROBE_OBJECT
        ),
        "reasoning_content_present": bool(reasoning_content),
        "reasoning_character_count": reasoning_character_count,
        "prompt_tokens": getattr(
            usage,
            "prompt_tokens",
            None,
        ),
        "completion_tokens": getattr(
            usage,
            "completion_tokens",
            None,
        ),
        "total_tokens": getattr(
            usage,
            "total_tokens",
            None,
        ),
        "maximum_completion_tokens": (
            THINKING_CONTROL_PROBE_MAX_TOKENS
        ),
        "automatic_retries": UOA_AUTOMATIC_RETRIES,
        "error_type": None,
        "error_message": None,
    }

except Exception as exc:
    probe_elapsed_seconds = time.perf_counter() - probe_start_time

    thinking_control_probe_result = {
        "probe_started_at_utc": probe_started_at_utc,
        "request_succeeded": False,
        "requested_model": UOA_GATEWAY_MODEL,
        "reported_model": None,
        "system_fingerprint": None,
        "finish_reason": None,
        "elapsed_seconds": probe_elapsed_seconds,
        "thinking_requested": "disabled",
        "visible_response": "",
        "visible_response_character_count": 0,
        "visible_json_parsed": False,
        "parsed_object_matches_expected": False,
        "reasoning_content_present": False,
        "reasoning_character_count": 0,
        "prompt_tokens": None,
        "completion_tokens": None,
        "total_tokens": None,
        "maximum_completion_tokens": (
            THINKING_CONTROL_PROBE_MAX_TOKENS
        ),
        "automatic_retries": UOA_AUTOMATIC_RETRIES,
        "error_type": type(exc).__name__,
        "error_message": str(exc),
    }

pd.Series(thinking_control_probe_result)

probe_started_at_utc                2026-09-02T21:05:25.203502+00:00
request_succeeded                                               True
requested_model                                           MiniMax-M3
reported_model                                            MiniMax-M3
system_fingerprint                              vllm-0.27.1-c607c424
finish_reason                                                   stop
elapsed_seconds                                             4.669341
thinking_requested                                          disabled
visible_response                          {"status":"ok","code":742}
visible_response_character_count                                  26
visible_json_parsed                                             True
parsed_object_matches_expected                                  True
reasoning_content_present                                       True
reasoning_character_count                                        311
prompt_tokens                     

### 1.2 Capability-gate decision

The following cell performs no network request. It converts the recorded probe metadata into an explicit blocking decision.

A passing decision means only that the gateway accepted the disabled-thinking request and returned the expected minimal response. It does not yet authorise the full 400-request batch. A paired research-record preflight remains required.

In [5]:
thinking_control_gate_checks = {
    "request_succeeded": (
        thinking_control_probe_result["request_succeeded"] is True
    ),
    "normal_finish": (
        thinking_control_probe_result["finish_reason"] == "stop"
    ),
    "visible_response_present": (
        thinking_control_probe_result[
            "visible_response_character_count"
        ]
        > 0
    ),
    "visible_json_parsed": (
        thinking_control_probe_result["visible_json_parsed"] is True
    ),
    "expected_object_returned": (
        thinking_control_probe_result[
            "parsed_object_matches_expected"
        ]
        is True
    ),
    "reasoning_content_absent": (
        thinking_control_probe_result[
            "reasoning_content_present"
        ]
        is False
    ),
    "reasoning_character_count_zero": (
        thinking_control_probe_result[
            "reasoning_character_count"
        ]
        == 0
    ),
}

thinking_control_gate_passed = all(
    thinking_control_gate_checks.values()
)

thinking_control_gate_summary = pd.Series(
    {
        **thinking_control_gate_checks,
        "all_checks_passed": thinking_control_gate_passed,
        "network_request_made_by_this_cell": False,
        "paired_preflight_authorised": (
            thinking_control_gate_passed
        ),
        "batch_inference_authorised": False,
    },
    name="value",
)

thinking_control_gate_summary

request_succeeded                     True
normal_finish                         True
visible_response_present              True
visible_json_parsed                   True
expected_object_returned              True
reasoning_content_absent             False
reasoning_character_count_zero       False
all_checks_passed                    False
network_request_made_by_this_cell    False
paired_preflight_authorised          False
batch_inference_authorised           False
Name: value, dtype: bool

### 1.3 Interpretation of residual reasoning content

The strict capability gate above did not pass because the response still contained 311 characters of provider-specific reasoning content, despite the request specifying disabled thinking.

This result must not be described as complete elimination of reasoning. It nevertheless differs materially from the behaviour observed in Notebook 05:

- the gateway accepted the provider-specific parameter without an API error;
- the request completed in approximately five seconds;
- the response ended normally with `finish_reason="stop"`;
- the expected visible JSON object was returned and parsed successfully;
- only 89 completion tokens were consumed; and
- the completion allowance was not exhausted.

The original zero-reasoning gate is retained unchanged as evidence that the provider did not produce a literally reasoning-free response.

For the research workflow, the operational requirement is not that the provider return exactly zero reasoning characters. The requirement is that the same inference setting produces complete, bounded and schema-valid visible responses for both representation conditions without repeatedly exhausting the completion budget.

A second, operational capability decision is therefore evaluated below. This decision does not claim that thinking was fully disabled. It records that the parameter was accepted and that the resulting response was operationally bounded enough to justify a paired research-record preflight.

No additional API request is made by the following decision cell.

In [6]:
# Preserve the earlier strict gate and its failed result. This second gate
# evaluates whether the observed behaviour is operationally usable for a
# controlled paired preflight, without claiming that reasoning was absent.
thinking_control_operational_checks = {
    "request_succeeded": (
        thinking_control_probe_result["request_succeeded"] is True
    ),
    "gateway_accepted_parameter": (
        thinking_control_probe_result["error_type"] is None
    ),
    "normal_finish": (
        thinking_control_probe_result["finish_reason"] == "stop"
    ),
    "visible_response_present": (
        thinking_control_probe_result[
            "visible_response_character_count"
        ]
        > 0
    ),
    "visible_json_parsed": (
        thinking_control_probe_result["visible_json_parsed"] is True
    ),
    "expected_object_returned": (
        thinking_control_probe_result[
            "parsed_object_matches_expected"
        ]
        is True
    ),
    "completion_budget_not_exhausted": (
        thinking_control_probe_result["completion_tokens"]
        < thinking_control_probe_result[
            "maximum_completion_tokens"
        ]
    ),
}

thinking_control_operationally_usable = all(
    thinking_control_operational_checks.values()
)

thinking_control_operational_summary = pd.Series(
    {
        **thinking_control_operational_checks,
        # Residual reasoning is reported explicitly rather than treated as
        # evidence that the provider returned a reasoning-free response.
        "reasoning_fully_disabled": (
            thinking_control_probe_result[
                "reasoning_character_count"
            ]
            == 0
        ),
        "residual_reasoning_observed": (
            thinking_control_probe_result[
                "reasoning_character_count"
            ]
            > 0
        ),
        "residual_reasoning_character_count": (
            thinking_control_probe_result[
                "reasoning_character_count"
            ]
        ),
        "operational_checks_passed": (
            thinking_control_operationally_usable
        ),
        "paired_preflight_authorised": (
            thinking_control_operationally_usable
        ),
        "batch_inference_authorised": False,
        "network_request_made_by_this_cell": False,
    },
    name="value",
)

thinking_control_operational_summary

request_succeeded                      True
gateway_accepted_parameter             True
normal_finish                          True
visible_response_present               True
visible_json_parsed                    True
expected_object_returned               True
completion_budget_not_exhausted        True
reasoning_fully_disabled              False
residual_reasoning_observed            True
residual_reasoning_character_count      311
operational_checks_passed              True
paired_preflight_authorised            True
batch_inference_authorised            False
network_request_made_by_this_cell     False
Name: value, dtype: object

## 2. Paired research-record preflight

The provider-independent task definition, system instruction, record delimiters and output schema remain unchanged from the locked protocol produced in Notebook 05.

The paired preflight uses the first validated sample in both representation conditions. The two requests therefore contain:

- the same underlying 46 feature–value pairs;
- the same system instruction;
- the same class definitions;
- the same output schema;
- the same model and provider endpoint;
- the same temperature and seed;
- the same thinking-control request;
- the same completion allowance; and
- the same client timeout.

Only the record representation differs.

The local sample identifier, feature-set identifier and canonical-payload hash are retained in the local execution record for auditability, but they are not included in the message sent to the LLM. Ground-truth labels are not loaded by this notebook.

A 2,048-token completion allowance is used for the paired preflight. This is substantially below the unsuccessful 4,096-token allowance used with default thinking in Notebook 05, while providing ample room for the required visible JSON object and any residual provider-specific reasoning.

Exactly two external requests are made: one for each representation condition. Automatic retries remain disabled.

In [7]:
COMMON_SYSTEM_INSTRUCTION = (
    llm_protocol["instructions"]["system_instruction"]
)
RECORD_OPENING_DELIMITER = (
    llm_protocol["instructions"]["record_opening_delimiter"]
)
RECORD_CLOSING_DELIMITER = (
    llm_protocol["instructions"]["record_closing_delimiter"]
)


def build_user_message(model_input: str) -> str:
    """
    Enclose one validated representation in the locked record delimiters.

    This function does not transform feature names or values and does not add
    a sample identifier, ground-truth label, dataset name, class balance,
    threshold, normal range or worked example.
    """
    if not isinstance(model_input, str):
        raise TypeError(
            "model_input must be a string, not "
            f"{type(model_input).__name__}."
        )

    if not model_input.strip():
        raise ValueError("model_input must not be empty.")

    if (
        RECORD_OPENING_DELIMITER in model_input
        or RECORD_CLOSING_DELIMITER in model_input
    ):
        raise ValueError(
            "model_input already contains a reserved record delimiter."
        )

    return (
        f"{RECORD_OPENING_DELIMITER}\n"
        f"{model_input}\n"
        f"{RECORD_CLOSING_DELIMITER}"
    )


def build_logical_request_contract(model_input: str) -> dict:
    """
    Build one provider-independent request contract.

    Provider-specific settings such as the model name, timeout, completion
    allowance and thinking control are deliberately added only when the
    external request is executed.
    """
    return {
        "system_instruction": COMMON_SYSTEM_INSTRUCTION,
        "user_message": build_user_message(model_input),
        "output_schema": llm_output_schema,
    }


preflight_sample_id = structured_records[0]["sample_id"]

assert preflight_sample_id == text_records[0]["sample_id"]
assert (
    structured_records[0]["canonical_payload_sha256"]
    == text_records[0]["canonical_payload_sha256"]
)

structured_preflight_contract = build_logical_request_contract(
    structured_records[0]["model_input"]
)
text_preflight_contract = build_logical_request_contract(
    text_records[0]["model_input"]
)

preflight_contract_summary = pd.Series(
    {
        "local_sample_id": preflight_sample_id,
        "feature_set_id": structured_records[0]["feature_set_id"],
        "canonical_payload_hashes_match": (
            structured_records[0]["canonical_payload_sha256"]
            == text_records[0]["canonical_payload_sha256"]
        ),
        "system_instructions_match": (
            structured_preflight_contract["system_instruction"]
            == text_preflight_contract["system_instruction"]
        ),
        "output_schemas_match": (
            structured_preflight_contract["output_schema"]
            == text_preflight_contract["output_schema"]
        ),
        "user_messages_differ": (
            structured_preflight_contract["user_message"]
            != text_preflight_contract["user_message"]
        ),
        "sample_id_sent_to_model": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

preflight_contract_summary

local_sample_id                    pilot_001
feature_set_id                    primary_46
canonical_payload_hashes_match          True
system_instructions_match               True
output_schemas_match                    True
user_messages_differ                    True
sample_id_sent_to_model                False
ground_truth_loaded                    False
network_request_made                   False
Name: value, dtype: object

### 2.1 Paired-preflight execution function

The following function translates one provider-independent request contract into one UoA-gateway request.

Each invocation:

- sends exactly one network-flow representation;
- uses the locked common system instruction and output schema;
- applies identical provider-specific settings;
- requests disabled thinking while acknowledging that residual reasoning may still be returned;
- makes at most one external request because automatic retries are disabled;
- records the requested and reported model identifiers;
- records completion status, latency and token usage;
- retains the visible response required for later validation;
- records only the character count of provider-specific reasoning and does not retain its text;
- stores a stable hash of the logical and provider-specific request content; and
- catches an API failure as data rather than automatically repeating the request.

The local execution record contains the sample identifier and condition so that results can later be joined and audited. These local metadata fields are not included in the messages sent to the LLM.

The response is validated at three levels:

1. completion validity — the request succeeded and ended normally;
2. schema validity — the visible response contains the required label and five citations; and
3. grounding validity — all cited feature names and values occur in the supplied record.

These checks are performed without loading the hidden ground-truth label.

In [8]:
PREFLIGHT_MAX_TOKENS = 2048
EXPECTED_CITATION_COUNT = 5
ALLOWED_PREDICTED_LABELS = {"Benign", "DoS"}
ALLOWED_CONDITIONS = {"structured", "deterministic_text"}


def stable_json_sha256(value: object) -> str:
    """
    Calculate a reproducible SHA-256 hash for a JSON-compatible object.

    Keys are sorted and unnecessary whitespace is removed so that the digest
    represents request content rather than Python dictionary display format.
    """
    canonical_json = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        canonical_json.encode("utf-8")
    ).hexdigest()


def canonical_json_scalar_to_text(value: object) -> str:
    """
    Convert a JSON scalar into the exact textual form expected in citations.

    Notebook 04 stores structured feature values as JSON scalars while the
    output schema requires `observed_value` to be a string. JSON-compatible
    serialisation preserves lowercase booleans, null and numeric formatting.
    Plain strings are returned without adding quotation marks.
    """
    if isinstance(value, str):
        return value

    if value is None or isinstance(value, (bool, int, float)):
        return json.dumps(
            value,
            ensure_ascii=False,
            allow_nan=False,
            separators=(",", ":"),
        )

    raise TypeError(
        "Structured feature values must be JSON scalar values; received "
        f"{type(value).__name__}."
    )


def build_grounding_reference(
    structured_model_input: str,
) -> dict[str, str]:
    """
    Recover the feature-name-to-value mapping from a structured input.

    This mapping is used only to check whether the LLM copied supplied
    features and values. It does not contain or use a ground-truth label.
    """
    parsed_record = json.loads(structured_model_input)

    if not isinstance(parsed_record, dict):
        raise ValueError(
            "The structured model input must decode to an object."
        )

    if set(parsed_record) != {"record_type", "features"}:
        raise ValueError(
            "The structured input contains unexpected top-level fields."
        )

    if parsed_record["record_type"] != "network_flow":
        raise ValueError(
            "The structured input has an unexpected record_type."
        )

    features = parsed_record["features"]

    if not isinstance(features, list) or len(features) != 46:
        raise ValueError(
            "The structured input must contain exactly 46 features."
        )

    grounding_reference: dict[str, str] = {}

    for position, feature in enumerate(features, start=1):
        if not isinstance(feature, dict):
            raise TypeError(
                f"Feature {position} is not a JSON object."
            )

        if set(feature) != {"name", "value"}:
            raise ValueError(
                f"Feature {position} contains unexpected fields."
            )

        feature_name = feature["name"]

        if not isinstance(feature_name, str) or not feature_name:
            raise ValueError(
                f"Feature {position} has an invalid name."
            )

        if feature_name in grounding_reference:
            raise ValueError(
                f"Duplicate feature name: {feature_name!r}."
            )

        grounding_reference[feature_name] = (
            canonical_json_scalar_to_text(feature["value"])
        )

    return grounding_reference


def validate_visible_research_response(
    visible_response: str,
    grounding_reference: dict[str, str],
) -> dict:
    """
    Validate one visible LLM response without using ground truth.

    The function checks JSON parsing, required structure, allowed predicted
    labels, five distinct citations, supported feature names and exact copied
    values. It does not assess whether the prediction itself is correct.
    """
    validation = {
        "visible_json_parsed": False,
        "schema_structure_valid": False,
        "predicted_label_valid": False,
        "citation_count_valid": False,
        "distinct_feature_names_valid": False,
        "supported_feature_names_valid": False,
        "observed_values_match": False,
        "grounding_valid": False,
        "overall_response_valid": False,
        "validation_error": None,
    }

    try:
        parsed_response = json.loads(visible_response)
    except json.JSONDecodeError as exc:
        validation["validation_error"] = (
            f"JSONDecodeError: {str(exc)[:300]}"
        )
        return validation

    validation["visible_json_parsed"] = True

    if (
        not isinstance(parsed_response, dict)
        or set(parsed_response)
        != {"predicted_label", "cited_features"}
    ):
        validation["validation_error"] = (
            "The response does not contain exactly the required "
            "top-level fields."
        )
        return validation

    predicted_label = parsed_response["predicted_label"]
    cited_features = parsed_response["cited_features"]

    validation["predicted_label_valid"] = (
        predicted_label in ALLOWED_PREDICTED_LABELS
    )

    if not isinstance(cited_features, list):
        validation["validation_error"] = (
            "cited_features is not a list."
        )
        return validation

    validation["citation_count_valid"] = (
        len(cited_features) == EXPECTED_CITATION_COUNT
    )

    citation_structure_valid = all(
        isinstance(citation, dict)
        and set(citation)
        == {"feature_name", "observed_value", "rationale"}
        and isinstance(citation["feature_name"], str)
        and isinstance(citation["observed_value"], str)
        and isinstance(citation["rationale"], str)
        and bool(citation["rationale"].strip())
        for citation in cited_features
    )

    validation["schema_structure_valid"] = (
        validation["predicted_label_valid"]
        and validation["citation_count_valid"]
        and citation_structure_valid
    )

    if not citation_structure_valid:
        validation["validation_error"] = (
            "At least one citation has an invalid structure."
        )
        return validation

    cited_feature_names = [
        citation["feature_name"]
        for citation in cited_features
    ]

    validation["distinct_feature_names_valid"] = (
        len(set(cited_feature_names))
        == EXPECTED_CITATION_COUNT
    )

    validation["supported_feature_names_valid"] = all(
        feature_name in grounding_reference
        for feature_name in cited_feature_names
    )

    validation["observed_values_match"] = all(
        citation["feature_name"] in grounding_reference
        and citation["observed_value"]
        == grounding_reference[citation["feature_name"]]
        for citation in cited_features
    )

    validation["grounding_valid"] = all(
        [
            validation["citation_count_valid"],
            validation["distinct_feature_names_valid"],
            validation["supported_feature_names_valid"],
            validation["observed_values_match"],
        ]
    )

    validation["overall_response_valid"] = (
        validation["schema_structure_valid"]
        and validation["grounding_valid"]
    )

    if not validation["overall_response_valid"]:
        validation["validation_error"] = (
            "The visible response parsed successfully but failed one or "
            "more schema or grounding checks."
        )

    return validation


preflight_grounding_reference = build_grounding_reference(
    structured_records[0]["model_input"]
)

assert len(preflight_grounding_reference) == 46

pd.Series(
    {
        "sample_id": preflight_sample_id,
        "grounding_reference_feature_count": len(
            preflight_grounding_reference
        ),
        "ground_truth_used": False,
        "network_request_made": False,
    },
    name="value",
)

sample_id                            pilot_001
grounding_reference_feature_count           46
ground_truth_used                        False
network_request_made                     False
Name: value, dtype: object

In [9]:
def run_uoa_research_request(
    *,
    request_contract: dict,
    condition: str,
    local_sample_id: str,
    canonical_payload_sha256: str,
    grounding_reference: dict[str, str],
) -> dict:
    """
    Execute one auditable UoA-gateway research request.

    Automatic retries are disabled at the client level, so one invocation
    makes at most one external request. Exceptions are converted into bounded
    local result records and are not automatically retried.
    """
    if condition not in ALLOWED_CONDITIONS:
        raise ValueError(
            f"Unsupported representation condition: {condition!r}."
        )

    if not isinstance(local_sample_id, str) or not local_sample_id:
        raise ValueError("local_sample_id must be a non-empty string.")

    # This object contains every material request component except the API key.
    # Its digest allows later detection of accidental request changes.
    request_content_for_hash = {
        "model": UOA_GATEWAY_MODEL,
        "messages": [
            {
                "role": "system",
                "content": request_contract["system_instruction"],
            },
            {
                "role": "user",
                "content": request_contract["user_message"],
            },
        ],
        "temperature": INFERENCE_TEMPERATURE,
        "seed": INFERENCE_SEED,
        "max_tokens": PREFLIGHT_MAX_TOKENS,
        "response_format": {
            "type": "json_schema",
            "json_schema": request_contract["output_schema"],
        },
        "extra_body": {
            "thinking": {
                "type": "disabled",
            }
        },
    }

    request_sha256 = stable_json_sha256(
        request_content_for_hash
    )
    request_started_at_utc = datetime.now(
        timezone.utc
    ).isoformat()
    request_start_time = time.perf_counter()

    try:
        response = uoa_gateway_client.chat.completions.create(
            model=UOA_GATEWAY_MODEL,
            messages=request_content_for_hash["messages"],
            temperature=INFERENCE_TEMPERATURE,
            seed=INFERENCE_SEED,
            max_tokens=PREFLIGHT_MAX_TOKENS,
            response_format=request_content_for_hash[
                "response_format"
            ],
            extra_body=request_content_for_hash["extra_body"],
        )

        elapsed_seconds = (
            time.perf_counter() - request_start_time
        )
        choice = response.choices[0]
        message = choice.message
        visible_response = (message.content or "").strip()

        # Provider-specific reasoning text is intentionally not persisted.
        # Only its presence and size are retained for operational analysis.
        reasoning_content = getattr(
            message,
            "reasoning_content",
            None,
        )
        reasoning_character_count = len(
            reasoning_content or ""
        )

        usage = response.usage

        response_validation = (
            validate_visible_research_response(
                visible_response,
                grounding_reference,
            )
        )

        return {
            "sample_id": local_sample_id,
            "condition": condition,
            "feature_set_id": "primary_46",
            "canonical_payload_sha256": (
                canonical_payload_sha256
            ),
            "request_sha256": request_sha256,
            "request_started_at_utc": (
                request_started_at_utc
            ),
            "request_succeeded": True,
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": getattr(
                response,
                "model",
                None,
            ),
            "system_fingerprint": getattr(
                response,
                "system_fingerprint",
                None,
            ),
            "response_id": getattr(
                response,
                "id",
                None,
            ),
            "finish_reason": choice.finish_reason,
            "elapsed_seconds": elapsed_seconds,
            "visible_response": visible_response,
            "visible_response_character_count": len(
                visible_response
            ),
            "reasoning_content_present": bool(
                reasoning_content
            ),
            "reasoning_character_count": (
                reasoning_character_count
            ),
            "prompt_tokens": getattr(
                usage,
                "prompt_tokens",
                None,
            ),
            "completion_tokens": getattr(
                usage,
                "completion_tokens",
                None,
            ),
            "total_tokens": getattr(
                usage,
                "total_tokens",
                None,
            ),
            "temperature": INFERENCE_TEMPERATURE,
            "seed": INFERENCE_SEED,
            "maximum_completion_tokens": (
                PREFLIGHT_MAX_TOKENS
            ),
            "thinking_requested": "disabled",
            "client_timeout_seconds": (
                UOA_CLIENT_TIMEOUT_SECONDS
            ),
            "automatic_retries": (
                UOA_AUTOMATIC_RETRIES
            ),
            **response_validation,
            "error_type": None,
            "error_message": None,
        }

    except Exception as exc:
        elapsed_seconds = (
            time.perf_counter() - request_start_time
        )

        return {
            "sample_id": local_sample_id,
            "condition": condition,
            "feature_set_id": "primary_46",
            "canonical_payload_sha256": (
                canonical_payload_sha256
            ),
            "request_sha256": request_sha256,
            "request_started_at_utc": (
                request_started_at_utc
            ),
            "request_succeeded": False,
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": None,
            "system_fingerprint": None,
            "response_id": None,
            "finish_reason": None,
            "elapsed_seconds": elapsed_seconds,
            "visible_response": "",
            "visible_response_character_count": 0,
            "reasoning_content_present": False,
            "reasoning_character_count": 0,
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "temperature": INFERENCE_TEMPERATURE,
            "seed": INFERENCE_SEED,
            "maximum_completion_tokens": (
                PREFLIGHT_MAX_TOKENS
            ),
            "thinking_requested": "disabled",
            "client_timeout_seconds": (
                UOA_CLIENT_TIMEOUT_SECONDS
            ),
            "automatic_retries": (
                UOA_AUTOMATIC_RETRIES
            ),
            "visible_json_parsed": False,
            "schema_structure_valid": False,
            "predicted_label_valid": False,
            "citation_count_valid": False,
            "distinct_feature_names_valid": False,
            "supported_feature_names_valid": False,
            "observed_values_match": False,
            "grounding_valid": False,
            "overall_response_valid": False,
            "validation_error": None,
            "error_type": type(exc).__name__,
            "error_message": str(exc)[:500],
        }

### 2.2 Execute one paired preflight

The following cell makes exactly two external requests.

The structured request is executed first, followed by the deterministic-text request. This ordering is used only for operational preflight and is not treated as an experimental effect. The formal batch will use a predefined counterbalanced request order to reduce systematic time or ordering effects.

The cell contains a kernel-level guard against accidental duplicate execution. Both requests use the same sample, model, inference settings, output schema, completion allowance and grounding reference.

In [10]:
if not thinking_control_operationally_usable:
    raise RuntimeError(
        "The operational thinking-control gate did not pass. "
        "Paired preflight is not authorised."
    )

if "paired_preflight_results" in globals():
    raise RuntimeError(
        "The paired research-record preflight has already been executed "
        "in this kernel. Do not repeat quota-consuming requests accidentally."
    )

preflight_payload_sha256 = structured_records[0][
    "canonical_payload_sha256"
]

paired_preflight_results = [
    run_uoa_research_request(
        request_contract=structured_preflight_contract,
        condition="structured",
        local_sample_id=preflight_sample_id,
        canonical_payload_sha256=preflight_payload_sha256,
        grounding_reference=preflight_grounding_reference,
    ),
    run_uoa_research_request(
        request_contract=text_preflight_contract,
        condition="deterministic_text",
        local_sample_id=preflight_sample_id,
        canonical_payload_sha256=preflight_payload_sha256,
        grounding_reference=preflight_grounding_reference,
    ),
]

paired_preflight_summary_columns = [
    "sample_id",
    "condition",
    "request_succeeded",
    "requested_model",
    "reported_model",
    "system_fingerprint",
    "finish_reason",
    "elapsed_seconds",
    "visible_response_character_count",
    "reasoning_content_present",
    "reasoning_character_count",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "visible_json_parsed",
    "schema_structure_valid",
    "grounding_valid",
    "overall_response_valid",
    "error_type",
    "error_message",
]

paired_preflight_summary = pd.DataFrame(
    paired_preflight_results
)[paired_preflight_summary_columns]

display(paired_preflight_summary)

,sample_id,condition,request_succeeded,requested_model,reported_model,system_fingerprint,finish_reason,elapsed_seconds,visible_response_character_count,reasoning_content_present,reasoning_character_count,prompt_tokens,completion_tokens,total_tokens,visible_json_parsed,schema_structure_valid,grounding_valid,overall_response_valid,error_type,error_message
0,pilot_001,structured,True,MiniMax-M3,MiniMax-M3,vllm-0.27.1-c607c424,length,23.697980,1947,True,5127,1380,2048,3428,False,False,False,False,None,None
1,pilot_001,deterministic_text,True,MiniMax-M3,MiniMax-M3,vllm-0.27.1-c607c424,length,23.533079,0,True,7583,1358,2048,3406,False,False,False,False,None,None


In [11]:
for result in paired_preflight_results:
    print("=" * 80)
    print(f"Condition: {result['condition']}")
    print(f"Sample: {result['sample_id']}")
    print(f"Finish reason: {result['finish_reason']}")
    print(f"Overall response valid: {result['overall_response_valid']}")
    print("-" * 80)

    if result["visible_response"]:
        try:
            parsed_response = json.loads(
                result["visible_response"]
            )
            print(
                json.dumps(
                    parsed_response,
                    ensure_ascii=False,
                    indent=2,
                )
            )
        except json.JSONDecodeError:
            print(result["visible_response"])
    else:
        print(
            "No visible response was returned. "
            f"Error: {result['error_type']} — "
            f"{result['error_message']}"
        )

Condition: structured
Sample: pilot_001
Finish reason: length
Overall response valid: False
--------------------------------------------------------------------------------
{
  "cited_features": [
    {
      "feature_name": "IN_PKTS",
      "observed_value":   "{\"record_type\":\"network_flow\",\"features\":[{\"name\":\"L4_SRC_PORT\",\"value\":47350},{\"name\":\"L4_DST_PORT\",\"value\":1581},{\"name\":\"PROTOCOL\",\"value\":6},{\"name\":\"L7_PROTO\",\"value\":0},{\"name\":\"IN_BYTES\",\"value\":492},{\"name\":\"IN_PKTS\",\"value\":10},{\"name\":\"OUT_BYTES\",\"value\":504},{\"name\":\"OUT_PKTS\",\"value\":10},{\"name\":\"TCP_FLAGS\",\"value\":19},{\"name\":\"CLIENT_TCP_FLAGS\",\"value\":19},{\"name\":\"SERVER_TCP_FLAGS\",\"value\":19},{\"name\":\"FLOW_DURATION_MILLISECONDS\",\"value\":857},{\"name\":\"DURATION_IN\",\"value\":857},{\"name\":\"DURATION_OUT\",\"value\":788},{\"name\":\"MIN_TTL\",\"value\":254},{\"name\":\"MAX_TTL\",\"value\":255},{\"name\":\"LONGEST_FLOW_PKT\",\"value\":

In [12]:
# This schema is deliberately minimal. It tests endpoint and structured-output
# compatibility without transmitting any research data.
RESPONSES_API_PROBE_FORMAT = {
    "type": "json_schema",
    "name": "responses_api_probe",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "status": {
                "type": "string",
                "enum": ["ok"],
            },
            "code": {
                "type": "integer",
                "enum": [742],
            },
        },
        "required": [
            "status",
            "code",
        ],
        "additionalProperties": False,
    },
}

EXPECTED_RESPONSES_API_PROBE_OBJECT = {
    "status": "ok",
    "code": 742,
}

# If the endpoint behaves as documented, 128 output tokens should be more than
# sufficient for the expected 26-character JSON object.
RESPONSES_API_PROBE_MAX_OUTPUT_TOKENS = 128

# Prevent accidental duplicate execution in the current kernel.
if "responses_api_probe_result" in globals():
    raise RuntimeError(
        "The Responses-API capability probe has already been executed "
        "in this kernel. Do not repeat the external request accidentally."
    )

responses_probe_started_at_utc = datetime.now(
    timezone.utc
).isoformat()
responses_probe_start_time = time.perf_counter()

try:
    responses_probe_response = uoa_gateway_client.responses.create(
        model=UOA_GATEWAY_MODEL,
        input=[
            {
                "role": "user",
                "content": (
                    "Return an object with status 'ok' and code 742. "
                    "Return only the schema-conforming object."
                ),
            }
        ],
        text={
            "format": RESPONSES_API_PROBE_FORMAT,
        },
        max_output_tokens=(
            RESPONSES_API_PROBE_MAX_OUTPUT_TOKENS
        ),

        # No `reasoning` parameter is included. Its omission is intentional:
        # the probe tests the Responses API's direct-answer behaviour.
    )

    responses_probe_elapsed_seconds = (
        time.perf_counter() - responses_probe_start_time
    )

    # The OpenAI SDK exposes the concatenated visible answer through
    # `output_text` when the response contains text output.
    responses_visible_output = (
        getattr(
            responses_probe_response,
            "output_text",
            "",
        )
        or ""
    ).strip()

    try:
        responses_parsed_output = json.loads(
            responses_visible_output
        )
        responses_visible_json_parsed = True
    except json.JSONDecodeError:
        responses_parsed_output = None
        responses_visible_json_parsed = False

    # Inspect output item types without retaining any potential reasoning text.
    responses_output_items = (
        getattr(
            responses_probe_response,
            "output",
            None,
        )
        or []
    )

    responses_output_item_types = [
        getattr(item, "type", type(item).__name__)
        for item in responses_output_items
    ]

    responses_reasoning_item_count = sum(
        item_type == "reasoning"
        for item_type in responses_output_item_types
    )

    responses_usage = getattr(
        responses_probe_response,
        "usage",
        None,
    )

    responses_api_probe_result = {
        "probe_started_at_utc": (
            responses_probe_started_at_utc
        ),
        "request_succeeded": True,
        "endpoint_method": "responses.create",
        "requested_model": UOA_GATEWAY_MODEL,
        "reported_model": getattr(
            responses_probe_response,
            "model",
            None,
        ),
        "response_id": getattr(
            responses_probe_response,
            "id",
            None,
        ),
        "response_status": getattr(
            responses_probe_response,
            "status",
            None,
        ),
        "incomplete_details": str(
            getattr(
                responses_probe_response,
                "incomplete_details",
                None,
            )
        ),
        "elapsed_seconds": responses_probe_elapsed_seconds,
        "visible_output": responses_visible_output,
        "visible_output_character_count": len(
            responses_visible_output
        ),
        "visible_json_parsed": (
            responses_visible_json_parsed
        ),
        "parsed_object_matches_expected": (
            responses_parsed_output
            == EXPECTED_RESPONSES_API_PROBE_OBJECT
        ),
        "output_item_types": (
            responses_output_item_types
        ),
        "reasoning_item_count": (
            responses_reasoning_item_count
        ),
        "input_tokens": getattr(
            responses_usage,
            "input_tokens",
            None,
        ),
        "output_tokens": getattr(
            responses_usage,
            "output_tokens",
            None,
        ),
        "total_tokens": getattr(
            responses_usage,
            "total_tokens",
            None,
        ),
        "maximum_output_tokens": (
            RESPONSES_API_PROBE_MAX_OUTPUT_TOKENS
        ),
        "reasoning_parameter_supplied": False,
        "automatic_retries": UOA_AUTOMATIC_RETRIES,
        "research_data_transmitted": False,
        "ground_truth_loaded": False,
        "error_type": None,
        "error_message": None,
    }

except Exception as exc:
    responses_probe_elapsed_seconds = (
        time.perf_counter() - responses_probe_start_time
    )

    responses_api_probe_result = {
        "probe_started_at_utc": (
            responses_probe_started_at_utc
        ),
        "request_succeeded": False,
        "endpoint_method": "responses.create",
        "requested_model": UOA_GATEWAY_MODEL,
        "reported_model": None,
        "response_id": None,
        "response_status": None,
        "incomplete_details": None,
        "elapsed_seconds": responses_probe_elapsed_seconds,
        "visible_output": "",
        "visible_output_character_count": 0,
        "visible_json_parsed": False,
        "parsed_object_matches_expected": False,
        "output_item_types": [],
        "reasoning_item_count": 0,
        "input_tokens": None,
        "output_tokens": None,
        "total_tokens": None,
        "maximum_output_tokens": (
            RESPONSES_API_PROBE_MAX_OUTPUT_TOKENS
        ),
        "reasoning_parameter_supplied": False,
        "automatic_retries": UOA_AUTOMATIC_RETRIES,
        "research_data_transmitted": False,
        "ground_truth_loaded": False,
        "error_type": type(exc).__name__,
        "error_message": str(exc)[:500],
    }

pd.Series(responses_api_probe_result)

probe_started_at_utc                               2026-09-02T21:38:57.743384+00:00
request_succeeded                                                              True
endpoint_method                                                    responses.create
requested_model                                                          MiniMax-M3
reported_model                                                           MiniMax-M3
response_id                       resp_Hv9V3EEi2UWZaeHS9WcX_TirGsSYT_ATM7JyTbc5y...
response_status                                                           completed
incomplete_details                                                             None
elapsed_seconds                                                            4.691276
visible_output                                           {"status":"ok","code":742}
visible_output_character_count                                                   26
visible_json_parsed                                                         

## Responses API paired research-record preflight

The earlier Chat Completions paired preflight established that the gateway
accepted the locked research prompt and JSON Schema, but both representation
conditions exhausted the 2,048-token completion allowance. Provider-specific
reasoning consumed a substantial part of that allowance, leaving an incomplete
or absent visible response. Those failed requests are retained above as
diagnostic evidence and will not be overwritten or silently retried.

A minimal, non-research capability probe subsequently confirmed that the same
UoA gateway and `MiniMax-M3` model support the Responses API, including strict
JSON-Schema output. The next experiment therefore evaluates whether the
Responses API can return a complete research response for one matched record
under both input-representation conditions.

This is an endpoint-level recovery test, not a change to the research question.
The following elements remain identical across the paired conditions:

- model and gateway;
- underlying network-flow record;
- 46-feature canonical payload;
- system instruction;
- binary class definitions;
- output schema;
- completion allowance; and
- local schema and grounding validation.

Only the representation-specific user message differs. The local sample
identifier, dataset identity and ground-truth label are not transmitted.

The preflight uses an 8,192-token output allowance because earlier 2,048- and
4,096-token Chat Completions attempts were insufficient once provider reasoning
was included. A 300-second per-request timeout follows the observed latency of
the university-hosted model and bounds each request independently. Automatic
retries remain disabled.

No `temperature`, `seed` or `reasoning` parameter is supplied through the
Responses API at this stage because support for those provider-specific
parameters has not been established on this endpoint. Both representation
conditions therefore receive the same provider defaults. This limitation will
be recorded explicitly rather than assuming unsupported controls.

The next code cell only constructs and audits the two request payloads. It does
not make a network request.

In [13]:
# The Responses API counts provider reasoning and the visible answer within the
# same output allowance. Earlier 2,048- and 4,096-token Chat Completions
# attempts did not leave enough capacity for a complete visible research
# response, so the paired preflight uses a larger but still bounded allowance.
RESPONSES_PREFLIGHT_MAX_OUTPUT_TOKENS = 8192

# The university-hosted model can be slow for full research prompts. This
# timeout applies independently to each request and does not enable retries.
RESPONSES_PREFLIGHT_TIMEOUT_SECONDS = 300.0

# Convert the already locked output-schema configuration into the flattened
# format expected by `responses.create(text={"format": ...})`.
#
# `llm_output_schema` contains:
# - name: the schema name;
# - strict: whether additional output structures are prohibited; and
# - schema: the actual JSON Schema.
RESPONSES_RESEARCH_TEXT_FORMAT = {
    "type": "json_schema",
    "name": llm_output_schema["name"],
    "strict": llm_output_schema["strict"],
    "schema": llm_output_schema["schema"],
}

# Build the provider-specific Responses API payload for each condition from the
# provider-independent logical contracts prepared earlier. The local sample ID,
# canonical hash, condition name and ground truth are deliberately excluded
# from the messages sent to the model.
structured_responses_preflight_payload = {
    "model": UOA_GATEWAY_MODEL,
    "input": [
        {
            "role": "system",
            "content": structured_preflight_contract[
                "system_instruction"
            ],
        },
        {
            "role": "user",
            "content": structured_preflight_contract[
                "user_message"
            ],
        },
    ],
    "text": {
        "format": RESPONSES_RESEARCH_TEXT_FORMAT,
    },
    "max_output_tokens": (
        RESPONSES_PREFLIGHT_MAX_OUTPUT_TOKENS
    ),
}

text_responses_preflight_payload = {
    "model": UOA_GATEWAY_MODEL,
    "input": [
        {
            "role": "system",
            "content": text_preflight_contract[
                "system_instruction"
            ],
        },
        {
            "role": "user",
            "content": text_preflight_contract[
                "user_message"
            ],
        },
    ],
    "text": {
        "format": RESPONSES_RESEARCH_TEXT_FORMAT,
    },
    "max_output_tokens": (
        RESPONSES_PREFLIGHT_MAX_OUTPUT_TOKENS
    ),
}

# These assertions enforce the controlled comparison before any research
# record is transmitted. They verify that the paired requests have the same
# model, instruction, schema and output allowance, while their user messages
# differ because their representations differ.
assert (
    structured_responses_preflight_payload["model"]
    == text_responses_preflight_payload["model"]
)
assert (
    structured_responses_preflight_payload["input"][0]
    == text_responses_preflight_payload["input"][0]
)
assert (
    structured_responses_preflight_payload["text"]
    == text_responses_preflight_payload["text"]
)
assert (
    structured_responses_preflight_payload["max_output_tokens"]
    == text_responses_preflight_payload["max_output_tokens"]
)
assert (
    structured_responses_preflight_payload["input"][1]["content"]
    != text_responses_preflight_payload["input"][1]["content"]
)

# Confirm again that both representations refer to the same canonical record.
assert (
    structured_records[0]["canonical_payload_sha256"]
    == text_records[0]["canonical_payload_sha256"]
)
assert (
    structured_records[0]["sample_id"]
    == text_records[0]["sample_id"]
    == preflight_sample_id
)

# Hash each complete provider-specific payload. The hashes are expected to
# differ because the user messages use different representations. They allow
# us to identify exactly which request content produced each later response.
structured_responses_request_sha256 = stable_json_sha256(
    structured_responses_preflight_payload
)
text_responses_request_sha256 = stable_json_sha256(
    text_responses_preflight_payload
)

responses_preflight_preparation_summary = pd.Series(
    {
        "sample_id": preflight_sample_id,
        "model": UOA_GATEWAY_MODEL,
        "endpoint_method": "responses.create",
        "feature_set_id": structured_records[0]["feature_set_id"],
        "canonical_payload_hashes_match": (
            structured_records[0]["canonical_payload_sha256"]
            == text_records[0]["canonical_payload_sha256"]
        ),
        "system_instructions_match": (
            structured_responses_preflight_payload["input"][0]
            == text_responses_preflight_payload["input"][0]
        ),
        "output_formats_match": (
            structured_responses_preflight_payload["text"]
            == text_responses_preflight_payload["text"]
        ),
        "maximum_output_tokens_match": (
            structured_responses_preflight_payload[
                "max_output_tokens"
            ]
            == text_responses_preflight_payload[
                "max_output_tokens"
            ]
        ),
        "user_messages_differ": (
            structured_responses_preflight_payload["input"][1]
            != text_responses_preflight_payload["input"][1]
        ),
        "structured_user_message_characters": len(
            structured_responses_preflight_payload[
                "input"
            ][1]["content"]
        ),
        "text_user_message_characters": len(
            text_responses_preflight_payload[
                "input"
            ][1]["content"]
        ),
        "structured_request_sha256": (
            structured_responses_request_sha256
        ),
        "text_request_sha256": text_responses_request_sha256,
        "maximum_output_tokens": (
            RESPONSES_PREFLIGHT_MAX_OUTPUT_TOKENS
        ),
        "client_timeout_seconds": (
            RESPONSES_PREFLIGHT_TIMEOUT_SECONDS
        ),
        "temperature_supplied": False,
        "seed_supplied": False,
        "reasoning_parameter_supplied": False,
        "automatic_retries": UOA_AUTOMATIC_RETRIES,
        "sample_id_sent_to_model": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

responses_preflight_preparation_summary

sample_id                                                                     pilot_001
model                                                                        MiniMax-M3
endpoint_method                                                        responses.create
feature_set_id                                                               primary_46
canonical_payload_hashes_match                                                     True
system_instructions_match                                                          True
output_formats_match                                                               True
maximum_output_tokens_match                                                        True
user_messages_differ                                                               True
structured_user_message_characters                                                 1914
text_user_message_characters                                                       1891
structured_request_sha256       

### Execute the Responses API paired preflight

This preflight sends the two previously audited representations of
`pilot_001` to the same UoA-hosted `MiniMax-M3` model through the Responses
API. The structured request is executed first and the deterministic-text
request second. Each function invocation makes at most one external request;
automatic retries remain disabled.

The two conditions use the same:

- canonical network-flow record;
- 46-feature set;
- system instruction;
- binary classification task;
- strict JSON Schema;
- model identifier;
- endpoint;
- maximum output-token allowance; and
- per-request timeout.

Only the representation-specific user message differs.

The Responses API may return separate `reasoning` and `message` output items.
The experiment records the presence and count of reasoning items, but does not
retain private provider reasoning content. Only the visible model answer is
retained and evaluated.

Successful transport is not sufficient for the preflight to pass. Each visible
answer must also:

1. parse as JSON;
2. contain exactly the required response fields;
3. provide an allowed binary label;
4. cite exactly five distinct supplied features;
5. copy all cited feature names exactly; and
6. copy the corresponding observed values exactly.

These checks use only the supplied feature record. Ground truth remains
unloaded, so this stage evaluates response completeness and grounding rather
than predictive correctness.

This cell defines the execution function only. The following cell contains the
explicitly guarded pair of quota-consuming requests.

In [14]:
def run_uoa_responses_research_request(
    *,
    request_payload: dict,
    condition: str,
    local_sample_id: str,
    canonical_payload_sha256: str,
    grounding_reference: dict[str, str],
) -> dict:
    """
    Execute one auditable research request through the UoA Responses API.

    The function sends only the locked system instruction and one validated
    representation-specific user message. Local identifiers, dataset
    provenance and ground truth are not included in the transmitted payload.

    One invocation makes at most one external request because the underlying
    OpenAI client was configured with automatic retries disabled.
    """
    if condition not in ALLOWED_CONDITIONS:
        raise ValueError(
            f"Unsupported representation condition: {condition!r}."
        )

    if not isinstance(local_sample_id, str) or not local_sample_id:
        raise ValueError(
            "local_sample_id must be a non-empty string."
        )

    # Verify that the caller has not changed the locked provider settings.
    if request_payload["model"] != UOA_GATEWAY_MODEL:
        raise ValueError(
            "The request payload contains an unexpected model."
        )

    if (
        request_payload["max_output_tokens"]
        != RESPONSES_PREFLIGHT_MAX_OUTPUT_TOKENS
    ):
        raise ValueError(
            "The request payload contains an unexpected output allowance."
        )

    # This digest covers all transmitted research-request content except the
    # API key. It allows later verification that a recorded response belongs
    # to the exact request payload prepared above.
    request_sha256 = stable_json_sha256(request_payload)

    request_started_at_utc = datetime.now(
        timezone.utc
    ).isoformat()
    request_start_time = time.perf_counter()

    try:
        # Apply the longer timeout only to this request path. This preserves
        # the original client configuration and keeps automatic retries at 0.
        response = (
            uoa_gateway_client
            .with_options(
                timeout=RESPONSES_PREFLIGHT_TIMEOUT_SECONDS
            )
            .responses.create(**request_payload)
        )

        elapsed_seconds = (
            time.perf_counter() - request_start_time
        )

        # `output_text` concatenates visible text from message output items.
        # Reasoning output is not copied into this field.
        visible_response = (
            getattr(response, "output_text", "") or ""
        ).strip()

        output_items = getattr(response, "output", None) or []
        output_item_types = [
            getattr(item, "type", type(item).__name__)
            for item in output_items
        ]

        reasoning_item_count = sum(
            item_type == "reasoning"
            for item_type in output_item_types
        )

        usage = getattr(response, "usage", None)
        output_token_details = getattr(
            usage,
            "output_tokens_details",
            None,
        )

        # Validate the visible answer against the locked response structure
        # and the 46 feature-name/value pairs supplied in the record.
        response_validation = (
            validate_visible_research_response(
                visible_response,
                grounding_reference,
            )
        )

        incomplete_details = getattr(
            response,
            "incomplete_details",
            None,
        )

        return {
            "sample_id": local_sample_id,
            "condition": condition,
            "feature_set_id": "primary_46",
            "canonical_payload_sha256": (
                canonical_payload_sha256
            ),
            "request_sha256": request_sha256,
            "request_started_at_utc": (
                request_started_at_utc
            ),
            "request_succeeded": True,
            "endpoint_method": "responses.create",
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": getattr(
                response,
                "model",
                None,
            ),
            "response_id": getattr(
                response,
                "id",
                None,
            ),
            "response_status": getattr(
                response,
                "status",
                None,
            ),
            "incomplete_details": (
                None
                if incomplete_details is None
                else str(incomplete_details)[:500]
            ),
            "elapsed_seconds": elapsed_seconds,
            "visible_response": visible_response,
            "visible_response_character_count": len(
                visible_response
            ),
            "output_item_types": output_item_types,
            "reasoning_item_count": reasoning_item_count,
            "input_tokens": getattr(
                usage,
                "input_tokens",
                None,
            ),
            "output_tokens": getattr(
                usage,
                "output_tokens",
                None,
            ),
            "reasoning_tokens": getattr(
                output_token_details,
                "reasoning_tokens",
                None,
            ),
            "total_tokens": getattr(
                usage,
                "total_tokens",
                None,
            ),
            "maximum_output_tokens": (
                RESPONSES_PREFLIGHT_MAX_OUTPUT_TOKENS
            ),
            "temperature_supplied": False,
            "seed_supplied": False,
            "reasoning_parameter_supplied": False,
            "client_timeout_seconds": (
                RESPONSES_PREFLIGHT_TIMEOUT_SECONDS
            ),
            "automatic_retries": UOA_AUTOMATIC_RETRIES,
            "research_data_transmitted": True,
            "sample_id_sent_to_model": False,
            "ground_truth_loaded": False,
            **response_validation,
            "error_type": None,
            "error_message": None,
        }

    except Exception as exc:
        elapsed_seconds = (
            time.perf_counter() - request_start_time
        )

        # Transport and SDK failures are converted into bounded result records
        # so that the second paired condition can still be attempted and the
        # diagnostic evidence remains available.
        return {
            "sample_id": local_sample_id,
            "condition": condition,
            "feature_set_id": "primary_46",
            "canonical_payload_sha256": (
                canonical_payload_sha256
            ),
            "request_sha256": request_sha256,
            "request_started_at_utc": (
                request_started_at_utc
            ),
            "request_succeeded": False,
            "endpoint_method": "responses.create",
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": None,
            "response_id": None,
            "response_status": None,
            "incomplete_details": None,
            "elapsed_seconds": elapsed_seconds,
            "visible_response": "",
            "visible_response_character_count": 0,
            "output_item_types": [],
            "reasoning_item_count": 0,
            "input_tokens": None,
            "output_tokens": None,
            "reasoning_tokens": None,
            "total_tokens": None,
            "maximum_output_tokens": (
                RESPONSES_PREFLIGHT_MAX_OUTPUT_TOKENS
            ),
            "temperature_supplied": False,
            "seed_supplied": False,
            "reasoning_parameter_supplied": False,
            "client_timeout_seconds": (
                RESPONSES_PREFLIGHT_TIMEOUT_SECONDS
            ),
            "automatic_retries": UOA_AUTOMATIC_RETRIES,
            "research_data_transmitted": True,
            "sample_id_sent_to_model": False,
            "ground_truth_loaded": False,
            "visible_json_parsed": False,
            "schema_structure_valid": False,
            "predicted_label_valid": False,
            "citation_count_valid": False,
            "distinct_feature_names_valid": False,
            "supported_feature_names_valid": False,
            "observed_values_match": False,
            "grounding_valid": False,
            "overall_response_valid": False,
            "validation_error": None,
            "error_type": type(exc).__name__,
            "error_message": str(exc)[:500],
        }

In [15]:
# This explicit guard prevents accidental duplicate paid or quota-consuming
# execution within the current kernel. An intentional rerun requires a kernel
# restart or documented removal of this variable.
if "responses_paired_preflight_results" in globals():
    raise RuntimeError(
        "The Responses API paired preflight has already been executed "
        "in this kernel. Do not repeat the external requests accidentally."
    )

# Reconfirm the non-label grounding reference immediately before transmission.
assert len(preflight_grounding_reference) == 46
assert preflight_sample_id == "pilot_001"

preflight_canonical_payload_sha256 = structured_records[0][
    "canonical_payload_sha256"
]

# Run the two conditions sequentially. Sequential execution avoids introducing
# concurrency as an additional difference during this small capability test.
responses_paired_preflight_results = [
    run_uoa_responses_research_request(
        request_payload=(
            structured_responses_preflight_payload
        ),
        condition="structured",
        local_sample_id=preflight_sample_id,
        canonical_payload_sha256=(
            preflight_canonical_payload_sha256
        ),
        grounding_reference=(
            preflight_grounding_reference
        ),
    ),
    run_uoa_responses_research_request(
        request_payload=text_responses_preflight_payload,
        condition="deterministic_text",
        local_sample_id=preflight_sample_id,
        canonical_payload_sha256=(
            preflight_canonical_payload_sha256
        ),
        grounding_reference=(
            preflight_grounding_reference
        ),
    ),
]

# Display only bounded diagnostic columns. Full visible responses remain in
# memory for the next inspection cell but are not printed here.
responses_paired_preflight_summary_columns = [
    "sample_id",
    "condition",
    "request_succeeded",
    "response_status",
    "elapsed_seconds",
    "visible_response_character_count",
    "output_item_types",
    "reasoning_item_count",
    "input_tokens",
    "output_tokens",
    "reasoning_tokens",
    "total_tokens",
    "visible_json_parsed",
    "schema_structure_valid",
    "grounding_valid",
    "overall_response_valid",
    "error_type",
    "error_message",
]

responses_paired_preflight_summary = pd.DataFrame(
    responses_paired_preflight_results
)[responses_paired_preflight_summary_columns]

display(responses_paired_preflight_summary)

,sample_id,condition,request_succeeded,response_status,elapsed_seconds,visible_response_character_count,output_item_types,reasoning_item_count,input_tokens,output_tokens,reasoning_tokens,total_tokens,visible_json_parsed,schema_structure_valid,grounding_valid,overall_response_valid,error_type,error_message
0,pilot_001,structured,True,incomplete,86.192533,138,"[reasoning, message]",1,900,8192,0,9092,False,False,False,False,None,None
1,pilot_001,deterministic_text,True,incomplete,85.977203,438,"[reasoning, message]",1,892,8192,0,9084,False,False,False,False,None,None


### Responses API preflight outcome and backend decision

Both Responses API requests reached the configured 8,192-token output limit
and returned `incomplete` status. Although transport succeeded and both
responses contained a message output item, neither visible response was
complete JSON and neither passed schema or grounding validation.

The provider reported a `reasoning` output item for each condition but did not
populate a usable reasoning-token count. Consequently, the recorded value of
zero reasoning tokens must not be interpreted as evidence that reasoning was
absent.

This reproduces the operational problem previously observed through Chat
Completions at smaller completion allowances: the UoA-hosted `MiniMax-M3`
deployment consumes the available output budget before producing a complete
schema-conforming research response.

The failed requests are retained as backend-feasibility evidence. They are not
used as anomaly-detection observations, and they will not be silently retried.
The output allowance will not be increased again without first testing an
alternative execution backend, because doing so would increase latency and
quota consumption without evidence that another finite allowance would
complete reliably.

The next recovery step tests the locally configured OpenCode route to the same
UoA `MiniMax-M3` model. This changes the execution backend, not the model,
research record, feature set, task or paired-representation design. OpenCode is
first tested using a minimal prompt containing no research data.

For risk reduction, the OpenCode probe:

- uses `--pure` to disable external plugins;
- runs with an empty temporary working directory;
- does not use `--auto`;
- attaches no files;
- uses a 300-second subprocess timeout; and
- records raw JSON event types without retaining hidden reasoning content.

In [16]:
import subprocess
import tempfile


OPENCODE_EXECUTABLE = "/opt/homebrew/bin/opencode"
OPENCODE_VERSION = "1.18.22"
OPENCODE_UOA_MODEL = "uoa/MiniMax-M3"
OPENCODE_REQUEST_TIMEOUT_SECONDS = 300

# This minimal prompt contains no research record, feature value, sample
# identifier, dataset information or ground truth.
OPENCODE_MINIMAL_PROBE_PROMPT = (
    "Return exactly the text GATEWAY_OK and nothing else."
)


def parse_opencode_json_events(stdout_text: str) -> dict:
    """
    Parse OpenCode's JSON-lines event stream without retaining reasoning text.

    OpenCode's `--format json` option controls the CLI event envelope; it does
    not force the model's visible answer itself to follow a research JSON
    Schema. Visible text events are concatenated, while event-type and token
    metadata are retained for backend diagnostics.
    """
    parsed_events = []
    malformed_line_count = 0

    for line in stdout_text.splitlines():
        stripped = line.strip()

        if not stripped:
            continue

        try:
            event = json.loads(stripped)
        except json.JSONDecodeError:
            malformed_line_count += 1
            continue

        if isinstance(event, dict):
            parsed_events.append(event)

    event_types = [
        event.get("type")
        for event in parsed_events
    ]

    visible_text_parts = []

    for event in parsed_events:
        if event.get("type") != "text":
            continue

        # Different OpenCode versions may expose the visible text directly or
        # inside a nested `part` object. Support both forms defensively.
        direct_text = event.get("text")
        nested_part = event.get("part")

        if isinstance(direct_text, str):
            visible_text_parts.append(direct_text)
        elif (
            isinstance(nested_part, dict)
            and isinstance(nested_part.get("text"), str)
        ):
            visible_text_parts.append(nested_part["text"])

    visible_text = "".join(visible_text_parts).strip()

    return {
        "event_count": len(parsed_events),
        "event_types": event_types,
        "malformed_line_count": malformed_line_count,
        "visible_text": visible_text,
    }

In [17]:
if "opencode_minimal_probe_result" in globals():
    raise RuntimeError(
        "The OpenCode minimal probe has already been executed in this "
        "kernel. Do not repeat the external request accidentally."
    )

opencode_probe_started_at_utc = datetime.now(
    timezone.utc
).isoformat()
opencode_probe_start_time = time.perf_counter()

try:
    # An empty temporary directory limits incidental repository context.
    # It is automatically removed when the `with` block finishes.
    with tempfile.TemporaryDirectory(
        prefix="compsci742_opencode_probe_"
    ) as temporary_context_directory:
        completed_process = subprocess.run(
            [
                OPENCODE_EXECUTABLE,
                "run",
                OPENCODE_MINIMAL_PROBE_PROMPT,
                "--dir",
                temporary_context_directory,
                "--pure",
                "--model",
                OPENCODE_UOA_MODEL,
                "--format",
                "json",
            ],
            capture_output=True,
            text=True,
            timeout=OPENCODE_REQUEST_TIMEOUT_SECONDS,
            check=False,
        )

    opencode_probe_elapsed_seconds = (
        time.perf_counter() - opencode_probe_start_time
    )

    parsed_opencode_probe = parse_opencode_json_events(
        completed_process.stdout
    )

    opencode_minimal_probe_result = {
        "probe_started_at_utc": (
            opencode_probe_started_at_utc
        ),
        "request_completed": True,
        "executable": OPENCODE_EXECUTABLE,
        "opencode_version": OPENCODE_VERSION,
        "requested_model": OPENCODE_UOA_MODEL,
        "return_code": completed_process.returncode,
        "elapsed_seconds": opencode_probe_elapsed_seconds,
        "event_count": parsed_opencode_probe["event_count"],
        "event_types": parsed_opencode_probe["event_types"],
        "malformed_line_count": (
            parsed_opencode_probe["malformed_line_count"]
        ),
        "visible_text": parsed_opencode_probe["visible_text"],
        "visible_text_matches_expected": (
            parsed_opencode_probe["visible_text"]
            == "GATEWAY_OK"
        ),
        "stderr_present": bool(
            completed_process.stderr.strip()
        ),
        "stderr_excerpt": (
            completed_process.stderr.strip()[:500]
        ),
        "pure_mode_used": True,
        "empty_temporary_directory_used": True,
        "auto_approval_used": False,
        "files_attached": False,
        "maximum_model_output_tokens_supplied": False,
        "subprocess_timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "research_data_transmitted": False,
        "ground_truth_loaded": False,
        "error_type": None,
        "error_message": None,
    }

except subprocess.TimeoutExpired as exc:
    opencode_probe_elapsed_seconds = (
        time.perf_counter() - opencode_probe_start_time
    )

    opencode_minimal_probe_result = {
        "probe_started_at_utc": (
            opencode_probe_started_at_utc
        ),
        "request_completed": False,
        "executable": OPENCODE_EXECUTABLE,
        "opencode_version": OPENCODE_VERSION,
        "requested_model": OPENCODE_UOA_MODEL,
        "return_code": None,
        "elapsed_seconds": opencode_probe_elapsed_seconds,
        "event_count": 0,
        "event_types": [],
        "malformed_line_count": 0,
        "visible_text": "",
        "visible_text_matches_expected": False,
        "stderr_present": False,
        "stderr_excerpt": "",
        "pure_mode_used": True,
        "empty_temporary_directory_used": True,
        "auto_approval_used": False,
        "files_attached": False,
        "maximum_model_output_tokens_supplied": False,
        "subprocess_timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "research_data_transmitted": False,
        "ground_truth_loaded": False,
        "error_type": type(exc).__name__,
        "error_message": str(exc)[:500],
    }

except Exception as exc:
    opencode_probe_elapsed_seconds = (
        time.perf_counter() - opencode_probe_start_time
    )

    opencode_minimal_probe_result = {
        "probe_started_at_utc": (
            opencode_probe_started_at_utc
        ),
        "request_completed": False,
        "executable": OPENCODE_EXECUTABLE,
        "opencode_version": OPENCODE_VERSION,
        "requested_model": OPENCODE_UOA_MODEL,
        "return_code": None,
        "elapsed_seconds": opencode_probe_elapsed_seconds,
        "event_count": 0,
        "event_types": [],
        "malformed_line_count": 0,
        "visible_text": "",
        "visible_text_matches_expected": False,
        "stderr_present": False,
        "stderr_excerpt": "",
        "pure_mode_used": True,
        "empty_temporary_directory_used": True,
        "auto_approval_used": False,
        "files_attached": False,
        "maximum_model_output_tokens_supplied": False,
        "subprocess_timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "research_data_transmitted": False,
        "ground_truth_loaded": False,
        "error_type": type(exc).__name__,
        "error_message": str(exc)[:500],
    }

pd.Series(opencode_minimal_probe_result)

probe_started_at_utc                    2026-09-02T22:30:52.915436+00:00
request_completed                                                   True
executable                                    /opt/homebrew/bin/opencode
opencode_version                                                 1.18.22
requested_model                                           uoa/MiniMax-M3
return_code                                                            0
elapsed_seconds                                                 6.956136
event_count                                                            3
event_types                              [step_start, text, step_finish]
malformed_line_count                                                   0
visible_text                                                  GATEWAY_OK
visible_text_matches_expected                                       True
stderr_present                                                     False
stderr_excerpt                                     

### OpenCode minimal-probe outcome

The minimal OpenCode probe completed successfully through
`uoa/MiniMax-M3`. The process returned exit code zero, emitted the expected
`step_start`, `text` and `step_finish` JSON events, and produced exactly the
visible text `GATEWAY_OK`.

The probe confirms that the locally configured OpenCode route is operational
without attaching files, enabling automatic permission approval or
transmitting research data. It also confirms that the JSON-lines event parser
can recover visible model text from OpenCode version 1.18.22.

OpenCode does not receive a model-output token limit from this notebook.
Therefore, this backend provides a bounded-time but not notebook-imposed
bounded-token recovery path for the UoA `MiniMax-M3` deployment.

Two provider-specific adaptations remain necessary before a research-record
preflight:

1. the OpenCode CLI accepts one prompt rather than separate system and user
   messages, so the locked instruction and record must be combined using one
   deterministic template; and
2. `--format json` structures OpenCode's event stream but does not enforce the
   research response schema, so the locked JSON Schema must be included in the
   prompt and validated locally after the response is returned.

Both adaptations will be identical across the structured and
deterministic-text conditions.

In [18]:
# Reparse the already completed minimal-probe event stream. This cell makes no
# network request and contains no research data.
opencode_probe_events = []

for line in completed_process.stdout.splitlines():
    stripped = line.strip()

    if not stripped:
        continue

    event = json.loads(stripped)

    if isinstance(event, dict):
        opencode_probe_events.append(event)


def describe_json_structure(value):
    """
    Describe nested JSON field names and value types without reproducing
    arbitrary text content.

    This is used to inspect OpenCode event structure safely. String values are
    represented only by their type and character count.
    """
    if isinstance(value, dict):
        return {
            key: describe_json_structure(nested_value)
            for key, nested_value in value.items()
        }

    if isinstance(value, list):
        return {
            "type": "list",
            "length": len(value),
            "item_types": sorted(
                {
                    type(item).__name__
                    for item in value
                }
            ),
        }

    if isinstance(value, str):
        return {
            "type": "str",
            "character_count": len(value),
        }

    return {
        "type": type(value).__name__,
        "value": value,
    }


opencode_event_structure_summary = [
    {
        "event_number": event_number,
        "event_type": event.get("type"),
        "structure": describe_json_structure(event),
    }
    for event_number, event in enumerate(
        opencode_probe_events,
        start=1,
    )
]

opencode_event_structure_summary

[{'event_number': 1,
  'event_type': 'step_start',
  'structure': {'type': {'type': 'str', 'character_count': 10},
   'timestamp': {'type': 'int', 'value': 1788388259385},
   'sessionID': {'type': 'str', 'character_count': 30},
   'part': {'id': {'type': 'str', 'character_count': 30},
    'messageID': {'type': 'str', 'character_count': 30},
    'sessionID': {'type': 'str', 'character_count': 30},
    'type': {'type': 'str', 'character_count': 10}}}},
 {'event_number': 2,
  'event_type': 'text',
  'structure': {'type': {'type': 'str', 'character_count': 4},
   'timestamp': {'type': 'int', 'value': 1788388259837},
   'sessionID': {'type': 'str', 'character_count': 30},
   'part': {'id': {'type': 'str', 'character_count': 30},
    'messageID': {'type': 'str', 'character_count': 30},
    'sessionID': {'type': 'str', 'character_count': 30},
    'type': {'type': 'str', 'character_count': 4},
    'text': {'type': 'str', 'character_count': 12},
    'time': {'start': {'type': 'int', 'value': 17

### OpenCode research-prompt construction

Inspection of the successful minimal probe showed that OpenCode version 1.18.22
reports token usage under `step_finish.part.tokens`. The probe used 7,993 input
tokens despite receiving only a minimal user prompt, indicating that OpenCode
adds a substantial internal agent context even in `--pure` mode and with an
empty working directory.

Consequently, the OpenCode route is treated as a distinct execution backend,
not as a request-equivalent replacement for the direct API. Results obtained
through this route will be attributed to the OpenCode-backed
`uoa/MiniMax-M3` configuration.

The comparison between representation conditions remains controlled because
both conditions use the same OpenCode version, model route, command options,
empty-directory policy and deterministic prompt template. Only the enclosed
network-flow representation differs.

Unlike the direct API, the OpenCode CLI accepts one positional prompt and does
not expose a server-enforced JSON-Schema argument. The provider-independent
system instruction, representation-specific record and locked response schema
are therefore combined into a single deterministic prompt. The schema is
serialised identically for both conditions, and each returned visible answer
will be validated locally using the existing schema and grounding checks.

The following cell constructs and audits the paired prompts only. It makes no
external request.

In [19]:
# Version this template so that any later wording change can be identified and
# distinguished from the current preflight requests.
OPENCODE_PROMPT_TEMPLATE_VERSION = "0.1.0"

# Serialise only the actual JSON Schema. Compact deterministic serialisation
# avoids condition-dependent whitespace and keeps the same schema text in both
# prompts.
OPENCODE_RESEARCH_SCHEMA_TEXT = json.dumps(
    llm_output_schema["schema"],
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
)


def build_opencode_research_prompt(
    request_contract: dict,
) -> str:
    """
    Convert one provider-independent logical request into one OpenCode prompt.

    OpenCode accepts one positional prompt rather than distinct system and user
    messages. This deterministic wrapper preserves the locked instruction,
    supplies the locked JSON Schema explicitly and encloses exactly one
    validated representation-specific record.

    Local sample identifiers, canonical hashes, dataset provenance and ground
    truth are not included.
    """
    required_contract_fields = {
        "system_instruction",
        "user_message",
        "output_schema",
    }

    if set(request_contract) != required_contract_fields:
        raise ValueError(
            "The logical request contract contains unexpected fields."
        )

    if request_contract["output_schema"] != llm_output_schema:
        raise ValueError(
            "The logical request contract does not use the locked schema."
        )

    return (
        "<research_instruction>\n"
        f"{request_contract['system_instruction']}\n"
        "</research_instruction>\n\n"
        "<required_output_json_schema>\n"
        f"{OPENCODE_RESEARCH_SCHEMA_TEXT}\n"
        "</required_output_json_schema>\n\n"
        "<research_input>\n"
        f"{request_contract['user_message']}\n"
        "</research_input>\n\n"
        "Follow the research instruction using only the enclosed research "
        "input. Return only the JSON object required by the enclosed schema."
    )


structured_opencode_preflight_prompt = (
    build_opencode_research_prompt(
        structured_preflight_contract
    )
)

text_opencode_preflight_prompt = (
    build_opencode_research_prompt(
        text_preflight_contract
    )
)

# Build reference prompts with the representation-specific user message removed.
# If these are identical, every non-record component is identical across the
# two experimental conditions.
structured_contract_without_record = {
    **structured_preflight_contract,
    "user_message": "<REPRESENTATION_PLACEHOLDER>",
}

text_contract_without_record = {
    **text_preflight_contract,
    "user_message": "<REPRESENTATION_PLACEHOLDER>",
}

structured_opencode_prompt_skeleton = (
    build_opencode_research_prompt(
        structured_contract_without_record
    )
)

text_opencode_prompt_skeleton = (
    build_opencode_research_prompt(
        text_contract_without_record
    )
)

# Enforce the paired experimental controls before either prompt is sent.
assert (
    structured_opencode_prompt_skeleton
    == text_opencode_prompt_skeleton
)
assert (
    structured_opencode_preflight_prompt
    != text_opencode_preflight_prompt
)
assert (
    OPENCODE_RESEARCH_SCHEMA_TEXT
    in structured_opencode_preflight_prompt
)
assert (
    OPENCODE_RESEARCH_SCHEMA_TEXT
    in text_opencode_preflight_prompt
)
assert preflight_sample_id not in (
    structured_opencode_preflight_prompt
)
assert preflight_sample_id not in (
    text_opencode_preflight_prompt
)

# The private ground-truth file has not been opened in this notebook. These
# checks additionally ensure that common label/provenance field names have not
# been introduced into the provider-specific wrapper.
for prohibited_wrapper_text in [
    "sample_id",
    "canonical_payload_sha256",
    "dataset_identity",
    "ground_truth",
]:
    assert prohibited_wrapper_text not in (
        structured_opencode_prompt_skeleton
    )
    assert prohibited_wrapper_text not in (
        text_opencode_prompt_skeleton
    )

structured_opencode_prompt_sha256 = stable_json_sha256(
    {
        "template_version": OPENCODE_PROMPT_TEMPLATE_VERSION,
        "prompt": structured_opencode_preflight_prompt,
    }
)

text_opencode_prompt_sha256 = stable_json_sha256(
    {
        "template_version": OPENCODE_PROMPT_TEMPLATE_VERSION,
        "prompt": text_opencode_preflight_prompt,
    }
)

opencode_prompt_preparation_summary = pd.Series(
    {
        "sample_id": preflight_sample_id,
        "feature_set_id": "primary_46",
        "backend": "opencode",
        "opencode_version": OPENCODE_VERSION,
        "model_route": OPENCODE_UOA_MODEL,
        "template_version": (
            OPENCODE_PROMPT_TEMPLATE_VERSION
        ),
        "non_record_prompt_skeletons_match": (
            structured_opencode_prompt_skeleton
            == text_opencode_prompt_skeleton
        ),
        "full_prompts_differ": (
            structured_opencode_preflight_prompt
            != text_opencode_preflight_prompt
        ),
        "structured_prompt_characters": len(
            structured_opencode_preflight_prompt
        ),
        "text_prompt_characters": len(
            text_opencode_preflight_prompt
        ),
        "structured_prompt_sha256": (
            structured_opencode_prompt_sha256
        ),
        "text_prompt_sha256": (
            text_opencode_prompt_sha256
        ),
        "schema_in_structured_prompt": (
            OPENCODE_RESEARCH_SCHEMA_TEXT
            in structured_opencode_preflight_prompt
        ),
        "schema_in_text_prompt": (
            OPENCODE_RESEARCH_SCHEMA_TEXT
            in text_opencode_preflight_prompt
        ),
        "sample_id_sent_to_model": False,
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

opencode_prompt_preparation_summary

sample_id                                                                    pilot_001
feature_set_id                                                              primary_46
backend                                                                       opencode
opencode_version                                                               1.18.22
model_route                                                             uoa/MiniMax-M3
template_version                                                                 0.1.0
non_record_prompt_skeletons_match                                                 True
full_prompts_differ                                                               True
structured_prompt_characters                                                      4049
text_prompt_characters                                                            4026
structured_prompt_sha256             f49c1346affe94d3a4be85086cfc5f362c91bcbbfb992d...
text_prompt_sha256                   577d32

### Execute the OpenCode-backed paired preflight

The audited structured and deterministic-text prompts for `pilot_001` are now
submitted sequentially through the same OpenCode-backed
`uoa/MiniMax-M3` route.

Each condition runs in a newly created empty temporary directory with external
plugins disabled through `--pure`. No files are attached and automatic
permission approval is not enabled. The subprocess timeout is 300 seconds per
condition. The notebook does not impose a model-output token limit because the
OpenCode CLI does not expose one through this execution path.

OpenCode returns JSON-lines operational events rather than a schema-enforced
model response. The execution function therefore:

1. concatenates visible `text` events;
2. extracts finish reasons and token counts from
   `step_finish.part.tokens`;
3. does not retain hidden reasoning text;
4. validates the visible response locally against the locked response
   structure; and
5. validates cited feature names and values against the supplied 46-feature
   record without loading ground truth.

A completed subprocess is not automatically considered a valid research
observation. Both transport and local response validation must succeed before
the backend is authorised for batch inference.

In [20]:
OPENCODE_BUDGET_EXHAUSTED_MARKERS = [
    "insufficient_quota",
    "quota exceeded",
    "credit balance",
    "out of credits",
    "usage limit",
    "payment required",
]


def parse_opencode_research_events(
    stdout_text: str,
) -> dict:
    """
    Parse one OpenCode JSON-lines stream into visible output and diagnostics.

    Visible text events are concatenated in emitted order. Token counts and
    finish reasons are recovered from `step_finish.part`. Raw event streams and
    hidden reasoning content are not retained in the returned record.
    """
    parsed_events = []
    malformed_line_count = 0

    for line in stdout_text.splitlines():
        stripped = line.strip()

        if not stripped:
            continue

        try:
            event = json.loads(stripped)
        except json.JSONDecodeError:
            malformed_line_count += 1
            continue

        if isinstance(event, dict):
            parsed_events.append(event)

    event_types = [
        event.get("type")
        for event in parsed_events
    ]

    visible_text_parts = []
    finish_reasons = []

    input_tokens = 0
    output_tokens = 0
    reasoning_tokens = 0
    total_tokens = 0
    cache_write_tokens = 0
    cache_read_tokens = 0
    reported_cost = 0.0
    step_finish_count = 0

    for event in parsed_events:
        event_type = event.get("type")
        part = event.get("part")

        if event_type == "text":
            direct_text = event.get("text")

            if isinstance(direct_text, str):
                visible_text_parts.append(direct_text)
            elif (
                isinstance(part, dict)
                and isinstance(part.get("text"), str)
            ):
                visible_text_parts.append(part["text"])

        if event_type != "step_finish":
            continue

        step_finish_count += 1

        if not isinstance(part, dict):
            continue

        reason = part.get("reason")

        if reason is not None:
            finish_reasons.append(str(reason))

        tokens = part.get("tokens")

        if isinstance(tokens, dict):
            input_tokens += int(tokens.get("input") or 0)
            output_tokens += int(tokens.get("output") or 0)
            reasoning_tokens += int(
                tokens.get("reasoning") or 0
            )
            total_tokens += int(tokens.get("total") or 0)

            cache_tokens = tokens.get("cache")

            if isinstance(cache_tokens, dict):
                cache_write_tokens += int(
                    cache_tokens.get("write") or 0
                )
                cache_read_tokens += int(
                    cache_tokens.get("read") or 0
                )

        cost = part.get("cost")

        if isinstance(cost, (int, float)):
            reported_cost += float(cost)

    visible_text = "".join(visible_text_parts).strip()

    return {
        "event_count": len(parsed_events),
        "event_types": event_types,
        "malformed_line_count": malformed_line_count,
        "step_finish_count": step_finish_count,
        "finish_reasons": finish_reasons,
        "visible_text": visible_text,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": reasoning_tokens,
        "total_tokens": total_tokens,
        "cache_write_tokens": cache_write_tokens,
        "cache_read_tokens": cache_read_tokens,
        "reported_cost": reported_cost,
    }


def run_opencode_research_request(
    *,
    prompt: str,
    prompt_sha256: str,
    condition: str,
    local_sample_id: str,
    canonical_payload_sha256: str,
    grounding_reference: dict[str, str],
) -> dict:
    """
    Execute one auditable OpenCode-backed research request.

    The prompt is passed as one subprocess argument without shell evaluation.
    The request runs in a fresh empty temporary directory with `--pure`,
    without file attachments or automatic permission approval.
    """
    if condition not in ALLOWED_CONDITIONS:
        raise ValueError(
            f"Unsupported representation condition: {condition!r}."
        )

    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt must be a non-empty string.")

    if local_sample_id in prompt:
        raise ValueError(
            "The local sample identifier must not appear in the prompt."
        )

    expected_prompt_sha256 = stable_json_sha256(
        {
            "template_version": (
                OPENCODE_PROMPT_TEMPLATE_VERSION
            ),
            "prompt": prompt,
        }
    )

    if prompt_sha256 != expected_prompt_sha256:
        raise ValueError(
            "The supplied prompt hash does not match the prompt content."
        )

    # Hash the prompt together with all material CLI execution settings. This
    # identifies the complete provider-specific request configuration.
    execution_contract = {
        "backend": "opencode",
        "opencode_version": OPENCODE_VERSION,
        "executable": OPENCODE_EXECUTABLE,
        "model": OPENCODE_UOA_MODEL,
        "prompt_template_version": (
            OPENCODE_PROMPT_TEMPLATE_VERSION
        ),
        "prompt_sha256": prompt_sha256,
        "pure": True,
        "empty_temporary_directory": True,
        "files_attached": False,
        "auto_approval": False,
        "format": "json",
        "timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "maximum_model_output_tokens": None,
    }

    execution_contract_sha256 = stable_json_sha256(
        execution_contract
    )

    request_started_at_utc = datetime.now(
        timezone.utc
    ).isoformat()
    request_start_time = time.perf_counter()

    try:
        with tempfile.TemporaryDirectory(
            prefix="compsci742_opencode_research_"
        ) as temporary_context_directory:
            completed_process = subprocess.run(
                [
                    OPENCODE_EXECUTABLE,
                    "run",
                    prompt,
                    "--dir",
                    temporary_context_directory,
                    "--pure",
                    "--model",
                    OPENCODE_UOA_MODEL,
                    "--format",
                    "json",
                ],
                capture_output=True,
                text=True,
                timeout=OPENCODE_REQUEST_TIMEOUT_SECONDS,
                check=False,
            )

        elapsed_seconds = (
            time.perf_counter() - request_start_time
        )

        parsed_events = parse_opencode_research_events(
            completed_process.stdout
        )

        visible_response = parsed_events["visible_text"]

        response_validation = (
            validate_visible_research_response(
                visible_response,
                grounding_reference,
            )
        )

        diagnostic_text = (
            visible_response
            + "\n"
            + completed_process.stderr
        ).lower()

        budget_exhausted = any(
            marker in diagnostic_text
            for marker in OPENCODE_BUDGET_EXHAUSTED_MARKERS
        )

        return {
            "sample_id": local_sample_id,
            "condition": condition,
            "feature_set_id": "primary_46",
            "backend": "opencode",
            "opencode_version": OPENCODE_VERSION,
            "requested_model": OPENCODE_UOA_MODEL,
            "canonical_payload_sha256": (
                canonical_payload_sha256
            ),
            "prompt_template_version": (
                OPENCODE_PROMPT_TEMPLATE_VERSION
            ),
            "prompt_sha256": prompt_sha256,
            "execution_contract_sha256": (
                execution_contract_sha256
            ),
            "request_started_at_utc": (
                request_started_at_utc
            ),
            "request_completed": True,
            "return_code": completed_process.returncode,
            "elapsed_seconds": elapsed_seconds,
            "event_count": parsed_events["event_count"],
            "event_types": parsed_events["event_types"],
            "malformed_line_count": (
                parsed_events["malformed_line_count"]
            ),
            "step_finish_count": (
                parsed_events["step_finish_count"]
            ),
            "finish_reasons": (
                parsed_events["finish_reasons"]
            ),
            "visible_response": visible_response,
            "visible_response_character_count": len(
                visible_response
            ),
            "input_tokens": parsed_events["input_tokens"],
            "output_tokens": parsed_events["output_tokens"],
            "reasoning_tokens": (
                parsed_events["reasoning_tokens"]
            ),
            "total_tokens": parsed_events["total_tokens"],
            "cache_write_tokens": (
                parsed_events["cache_write_tokens"]
            ),
            "cache_read_tokens": (
                parsed_events["cache_read_tokens"]
            ),
            "reported_cost": parsed_events["reported_cost"],
            "budget_exhausted": budget_exhausted,
            "stderr_present": bool(
                completed_process.stderr.strip()
            ),
            "stderr_excerpt": (
                completed_process.stderr.strip()[:500]
            ),
            "pure_mode_used": True,
            "empty_temporary_directory_used": True,
            "auto_approval_used": False,
            "files_attached": False,
            "maximum_model_output_tokens_supplied": False,
            "subprocess_timeout_seconds": (
                OPENCODE_REQUEST_TIMEOUT_SECONDS
            ),
            "research_data_transmitted": True,
            "sample_id_sent_to_model": False,
            "ground_truth_loaded": False,
            **response_validation,
            "error_type": None,
            "error_message": None,
        }

    except subprocess.TimeoutExpired as exc:
        elapsed_seconds = (
            time.perf_counter() - request_start_time
        )

        return {
            "sample_id": local_sample_id,
            "condition": condition,
            "feature_set_id": "primary_46",
            "backend": "opencode",
            "opencode_version": OPENCODE_VERSION,
            "requested_model": OPENCODE_UOA_MODEL,
            "canonical_payload_sha256": (
                canonical_payload_sha256
            ),
            "prompt_template_version": (
                OPENCODE_PROMPT_TEMPLATE_VERSION
            ),
            "prompt_sha256": prompt_sha256,
            "execution_contract_sha256": (
                execution_contract_sha256
            ),
            "request_started_at_utc": (
                request_started_at_utc
            ),
            "request_completed": False,
            "return_code": None,
            "elapsed_seconds": elapsed_seconds,
            "event_count": 0,
            "event_types": [],
            "malformed_line_count": 0,
            "step_finish_count": 0,
            "finish_reasons": [],
            "visible_response": "",
            "visible_response_character_count": 0,
            "input_tokens": None,
            "output_tokens": None,
            "reasoning_tokens": None,
            "total_tokens": None,
            "cache_write_tokens": None,
            "cache_read_tokens": None,
            "reported_cost": None,
            "budget_exhausted": False,
            "stderr_present": False,
            "stderr_excerpt": "",
            "pure_mode_used": True,
            "empty_temporary_directory_used": True,
            "auto_approval_used": False,
            "files_attached": False,
            "maximum_model_output_tokens_supplied": False,
            "subprocess_timeout_seconds": (
                OPENCODE_REQUEST_TIMEOUT_SECONDS
            ),
            "research_data_transmitted": True,
            "sample_id_sent_to_model": False,
            "ground_truth_loaded": False,
            "visible_json_parsed": False,
            "schema_structure_valid": False,
            "predicted_label_valid": False,
            "citation_count_valid": False,
            "distinct_feature_names_valid": False,
            "supported_feature_names_valid": False,
            "observed_values_match": False,
            "grounding_valid": False,
            "overall_response_valid": False,
            "validation_error": None,
            "error_type": type(exc).__name__,
            "error_message": str(exc)[:500],
        }

    except Exception as exc:
        elapsed_seconds = (
            time.perf_counter() - request_start_time
        )

        return {
            "sample_id": local_sample_id,
            "condition": condition,
            "feature_set_id": "primary_46",
            "backend": "opencode",
            "opencode_version": OPENCODE_VERSION,
            "requested_model": OPENCODE_UOA_MODEL,
            "canonical_payload_sha256": (
                canonical_payload_sha256
            ),
            "prompt_template_version": (
                OPENCODE_PROMPT_TEMPLATE_VERSION
            ),
            "prompt_sha256": prompt_sha256,
            "execution_contract_sha256": (
                execution_contract_sha256
            ),
            "request_started_at_utc": (
                request_started_at_utc
            ),
            "request_completed": False,
            "return_code": None,
            "elapsed_seconds": elapsed_seconds,
            "event_count": 0,
            "event_types": [],
            "malformed_line_count": 0,
            "step_finish_count": 0,
            "finish_reasons": [],
            "visible_response": "",
            "visible_response_character_count": 0,
            "input_tokens": None,
            "output_tokens": None,
            "reasoning_tokens": None,
            "total_tokens": None,
            "cache_write_tokens": None,
            "cache_read_tokens": None,
            "reported_cost": None,
            "budget_exhausted": False,
            "stderr_present": False,
            "stderr_excerpt": "",
            "pure_mode_used": True,
            "empty_temporary_directory_used": True,
            "auto_approval_used": False,
            "files_attached": False,
            "maximum_model_output_tokens_supplied": False,
            "subprocess_timeout_seconds": (
                OPENCODE_REQUEST_TIMEOUT_SECONDS
            ),
            "research_data_transmitted": False,
            "sample_id_sent_to_model": False,
            "ground_truth_loaded": False,
            "visible_json_parsed": False,
            "schema_structure_valid": False,
            "predicted_label_valid": False,
            "citation_count_valid": False,
            "distinct_feature_names_valid": False,
            "supported_feature_names_valid": False,
            "observed_values_match": False,
            "grounding_valid": False,
            "overall_response_valid": False,
            "validation_error": None,
            "error_type": type(exc).__name__,
            "error_message": str(exc)[:500],
        }

In [21]:
if "opencode_paired_preflight_results" in globals():
    raise RuntimeError(
        "The OpenCode paired preflight has already been executed in this "
        "kernel. Do not repeat the external requests accidentally."
    )

assert len(preflight_grounding_reference) == 46
assert preflight_sample_id == "pilot_001"

opencode_preflight_canonical_payload_sha256 = (
    structured_records[0]["canonical_payload_sha256"]
)

opencode_paired_preflight_results = [
    run_opencode_research_request(
        prompt=structured_opencode_preflight_prompt,
        prompt_sha256=structured_opencode_prompt_sha256,
        condition="structured",
        local_sample_id=preflight_sample_id,
        canonical_payload_sha256=(
            opencode_preflight_canonical_payload_sha256
        ),
        grounding_reference=(
            preflight_grounding_reference
        ),
    ),
    run_opencode_research_request(
        prompt=text_opencode_preflight_prompt,
        prompt_sha256=text_opencode_prompt_sha256,
        condition="deterministic_text",
        local_sample_id=preflight_sample_id,
        canonical_payload_sha256=(
            opencode_preflight_canonical_payload_sha256
        ),
        grounding_reference=(
            preflight_grounding_reference
        ),
    ),
]

opencode_paired_preflight_summary_columns = [
    "sample_id",
    "condition",
    "request_completed",
    "return_code",
    "elapsed_seconds",
    "finish_reasons",
    "visible_response_character_count",
    "input_tokens",
    "output_tokens",
    "reasoning_tokens",
    "total_tokens",
    "budget_exhausted",
    "stderr_present",
    "visible_json_parsed",
    "schema_structure_valid",
    "grounding_valid",
    "overall_response_valid",
    "error_type",
    "error_message",
]

opencode_paired_preflight_summary = pd.DataFrame(
    opencode_paired_preflight_results
)[opencode_paired_preflight_summary_columns]

display(opencode_paired_preflight_summary)

,sample_id,condition,request_completed,return_code,elapsed_seconds,finish_reasons,visible_response_character_count,input_tokens,output_tokens,reasoning_tokens,total_tokens,budget_exhausted,stderr_present,visible_json_parsed,schema_structure_valid,grounding_valid,overall_response_valid,error_type,error_message
0,pilot_001,structured,True,0,21.449245,[stop],1104,8664,368,1043,10075,False,False,True,True,True,True,None,None
1,pilot_001,deterministic_text,True,0,24.221286,[stop],1224,8609,428,1471,10508,False,False,True,True,True,True,None,None


### OpenCode paired-preflight outcome

The OpenCode-backed paired preflight completed successfully for both
representations of `pilot_001`.

Both subprocesses:

- returned exit code zero;
- ended with the reported reason `stop`;
- completed within the 300-second timeout;
- produced parseable JSON;
- satisfied the required response structure;
- cited exactly five distinct supplied features;
- copied all cited feature names and observed values correctly; and
- passed the complete local response-validation gate.

The structured condition completed in approximately 21.45 seconds and used
10,075 total tokens. The deterministic-text condition completed in
approximately 24.22 seconds and used 10,508 total tokens. OpenCode's internal
agent context accounted for most input tokens, while provider reasoning was
reported separately from visible output.

This result establishes technical feasibility for the OpenCode-backed
`uoa/MiniMax-M3` route. It does not yet establish predictive performance,
because ground truth remains unloaded, and one paired record is insufficient
to assess backend reliability or representation effects.

Before authorising batch inference, the following local inspection compares the
two valid visible responses for prediction agreement, cited-feature overlap and
rank agreement. It makes no external request.

In [22]:
# Parse the two already validated visible responses. This cell makes no
# external request and does not load ground truth.
parsed_opencode_preflight_responses = {
    result["condition"]: json.loads(
        result["visible_response"]
    )
    for result in opencode_paired_preflight_results
}

structured_parsed_response = (
    parsed_opencode_preflight_responses["structured"]
)

text_parsed_response = (
    parsed_opencode_preflight_responses[
        "deterministic_text"
    ]
)

structured_citations = (
    structured_parsed_response["cited_features"]
)
text_citations = text_parsed_response["cited_features"]

structured_cited_feature_names = [
    citation["feature_name"]
    for citation in structured_citations
]

text_cited_feature_names = [
    citation["feature_name"]
    for citation in text_citations
]

structured_feature_set = set(
    structured_cited_feature_names
)
text_feature_set = set(
    text_cited_feature_names
)

cited_feature_intersection = (
    structured_feature_set
    & text_feature_set
)

cited_feature_union = (
    structured_feature_set
    | text_feature_set
)

# Set-based overlap ignores ranking. With five citations in each condition,
# the Jaccard score is intersection size divided by union size.
preflight_feature_jaccard = (
    len(cited_feature_intersection)
    / len(cited_feature_union)
)

# Positional agreement measures how many of the five ranks contain the same
# feature in both representations.
preflight_exact_rank_matches = sum(
    structured_feature_name == text_feature_name
    for structured_feature_name, text_feature_name in zip(
        structured_cited_feature_names,
        text_cited_feature_names,
        strict=True,
    )
)

opencode_preflight_response_comparison = pd.Series(
    {
        "sample_id": preflight_sample_id,
        "structured_predicted_label": (
            structured_parsed_response["predicted_label"]
        ),
        "text_predicted_label": (
            text_parsed_response["predicted_label"]
        ),
        "predicted_labels_agree": (
            structured_parsed_response["predicted_label"]
            == text_parsed_response["predicted_label"]
        ),
        "structured_cited_features": (
            structured_cited_feature_names
        ),
        "text_cited_features": (
            text_cited_feature_names
        ),
        "shared_cited_feature_count": len(
            cited_feature_intersection
        ),
        "shared_cited_features": sorted(
            cited_feature_intersection
        ),
        "cited_feature_union_count": len(
            cited_feature_union
        ),
        "top_5_jaccard_similarity": (
            preflight_feature_jaccard
        ),
        "exact_rank_match_count": (
            preflight_exact_rank_matches
        ),
        "both_responses_schema_valid": all(
            result["schema_structure_valid"]
            for result in opencode_paired_preflight_results
        ),
        "both_responses_grounding_valid": all(
            result["grounding_valid"]
            for result in opencode_paired_preflight_results
        ),
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

opencode_preflight_response_comparison

sample_id                                                                 pilot_001
structured_predicted_label                                                   Benign
text_predicted_label                                                         Benign
predicted_labels_agree                                                         True
structured_cited_features         [IN_PKTS, TCP_FLAGS, FLOW_DURATION_MILLISECOND...
text_cited_features               [TCP_FLAGS, IN_PKTS, OUT_PKTS, SRC_TO_DST_AVG_...
shared_cited_feature_count                                                        3
shared_cited_features              [FLOW_DURATION_MILLISECONDS, IN_PKTS, TCP_FLAGS]
cited_feature_union_count                                                         7
top_5_jaccard_similarity                                                   0.428571
exact_rank_match_count                                                            0
both_responses_schema_valid                                                 

### Persist the paired-preflight evidence

The first valid OpenCode-backed paired responses are persisted before any
additional external requests are attempted. This protects the original
preflight evidence from kernel loss and prevents later batch logic from
silently replacing it.

The persisted result file contains one record per representation condition,
including:

- request and prompt hashes;
- backend and model identifiers;
- timestamps and elapsed time;
- OpenCode event diagnostics;
- reported token usage;
- the visible JSON response;
- local schema and grounding-validation results; and
- bounded error fields.

The file does not contain an API key, OpenCode provider credentials, hidden
reasoning content or ground truth.

A separate manifest records the relationship between the two responses,
the common canonical payload, protocol configuration and generated file
digest. Existing files are never overwritten by this cell.

In [23]:
PREFLIGHT_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "preflight"
    / "primary_46"
)

OPENCODE_PREFLIGHT_RESULTS_PATH = (
    PREFLIGHT_RESULTS_DIR
    / "opencode_paired_preflight_results.jsonl"
)

OPENCODE_PREFLIGHT_MANIFEST_PATH = (
    PREFLIGHT_RESULTS_DIR
    / "opencode_paired_preflight_manifest.json"
)


def sha256_file(path: Path) -> str:
    """Calculate the SHA-256 digest of one file's exact bytes."""
    return hashlib.sha256(path.read_bytes()).hexdigest()


def write_new_text_file(
    path: Path,
    content: str,
) -> None:
    """
    Write a new UTF-8 text file atomically without overwriting evidence.

    Content is first written to a sibling temporary file and then moved into
    place. Refusing to overwrite an existing target prevents accidental
    replacement of an earlier experiment.
    """
    if path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing evidence file: {path}"
        )

    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    if temporary_path.exists():
        raise FileExistsError(
            "A temporary evidence file already exists: "
            f"{temporary_path}"
        )

    temporary_path.write_text(
        content,
        encoding="utf-8",
    )

    temporary_path.replace(path)


# Confirm that exactly one structured and one deterministic-text result are
# being saved and that both belong to the same canonical network-flow payload.
assert len(opencode_paired_preflight_results) == 2
assert {
    result["condition"]
    for result in opencode_paired_preflight_results
} == {
    "structured",
    "deterministic_text",
}
assert len(
    {
        result["canonical_payload_sha256"]
        for result in opencode_paired_preflight_results
    }
) == 1
assert all(
    result["overall_response_valid"]
    for result in opencode_paired_preflight_results
)
assert all(
    not result["ground_truth_loaded"]
    for result in opencode_paired_preflight_results
)

# JSON Lines keeps the two condition-level records independent and permits
# streaming reads during later batch-analysis work.
opencode_preflight_results_jsonl = (
    "\n".join(
        json.dumps(
            result,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        for result in opencode_paired_preflight_results
    )
    + "\n"
)

write_new_text_file(
    OPENCODE_PREFLIGHT_RESULTS_PATH,
    opencode_preflight_results_jsonl,
)

# Hash the completed result file so the manifest can detect any later change
# to the stored responses or metadata.
opencode_preflight_results_file_sha256 = sha256_file(
    OPENCODE_PREFLIGHT_RESULTS_PATH
)

opencode_preflight_manifest = {
    "artifact_type": "paired_preflight_manifest",
    "artifact_version": "0.1.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "experiment_scope": "single_record_backend_preflight",
    "sample_count": 1,
    "request_count": 2,
    "sample_id": preflight_sample_id,
    "conditions": [
        "structured",
        "deterministic_text",
    ],
    "feature_set_id": "primary_46",
    "feature_count": 46,
    "backend": {
        "name": "opencode",
        "version": OPENCODE_VERSION,
        "model_route": OPENCODE_UOA_MODEL,
        "pure_mode": True,
        "empty_temporary_directory": True,
        "auto_approval": False,
        "files_attached": False,
        "subprocess_timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "maximum_model_output_tokens_supplied": False,
    },
    "prompt_template": {
        "version": OPENCODE_PROMPT_TEMPLATE_VERSION,
        "structured_prompt_sha256": (
            structured_opencode_prompt_sha256
        ),
        "deterministic_text_prompt_sha256": (
            text_opencode_prompt_sha256
        ),
    },
    "canonical_payload_sha256": (
        opencode_preflight_canonical_payload_sha256
    ),
    "protocol": {
        "path": PROTOCOL_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "file_sha256": sha256_file(PROTOCOL_PATH),
    },
    "output_schema": {
        "path": OUTPUT_SCHEMA_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "file_sha256": sha256_file(
            OUTPUT_SCHEMA_PATH
        ),
    },
    "validation": {
        "both_requests_completed": all(
            result["request_completed"]
            for result in opencode_paired_preflight_results
        ),
        "both_return_codes_zero": all(
            result["return_code"] == 0
            for result in opencode_paired_preflight_results
        ),
        "both_schema_valid": all(
            result["schema_structure_valid"]
            for result in opencode_paired_preflight_results
        ),
        "both_grounding_valid": all(
            result["grounding_valid"]
            for result in opencode_paired_preflight_results
        ),
        "both_overall_valid": all(
            result["overall_response_valid"]
            for result in opencode_paired_preflight_results
        ),
        "ground_truth_loaded": False,
    },
    "preflight_observation": {
        "predicted_labels_agree": (
            structured_parsed_response["predicted_label"]
            == text_parsed_response["predicted_label"]
        ),
        "shared_cited_feature_count": len(
            cited_feature_intersection
        ),
        "top_5_jaccard_similarity": (
            preflight_feature_jaccard
        ),
        "exact_rank_match_count": (
            preflight_exact_rank_matches
        ),
        "interpretation_status": (
            "technical_preflight_only"
        ),
    },
    "result_file": {
        "path": OPENCODE_PREFLIGHT_RESULTS_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "sha256": (
            opencode_preflight_results_file_sha256
        ),
    },
}

write_new_text_file(
    OPENCODE_PREFLIGHT_MANIFEST_PATH,
    json.dumps(
        opencode_preflight_manifest,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )
    + "\n",
)

preflight_persistence_summary = pd.Series(
    {
        "results_file": (
            OPENCODE_PREFLIGHT_RESULTS_PATH
            .relative_to(PROJECT_ROOT)
            .as_posix()
        ),
        "manifest_file": (
            OPENCODE_PREFLIGHT_MANIFEST_PATH
            .relative_to(PROJECT_ROOT)
            .as_posix()
        ),
        "saved_result_records": len(
            opencode_paired_preflight_results
        ),
        "results_file_sha256": (
            opencode_preflight_results_file_sha256
        ),
        "manifest_file_sha256": sha256_file(
            OPENCODE_PREFLIGHT_MANIFEST_PATH
        ),
        "both_overall_valid": (
            opencode_preflight_manifest[
                "validation"
            ]["both_overall_valid"]
        ),
        "ground_truth_saved": False,
        "reasoning_content_saved": False,
        "existing_files_overwritten": False,
        "network_request_made": False,
    },
    name="value",
)

preflight_persistence_summary

results_file                  results/preflight/primary_46/opencode_paired_p...
manifest_file                 results/preflight/primary_46/opencode_paired_p...
saved_result_records                                                          2
results_file_sha256           556dd921b69ed51d8edf339adb67845e4ca891155d8564...
manifest_file_sha256          d5b814d0faca2d2bf1e7142febbfcb1d2625d73c48e141...
both_overall_valid                                                         True
ground_truth_saved                                                        False
reasoning_content_saved                                                   False
existing_files_overwritten                                                False
network_request_made                                                      False
Name: value, dtype: object

## Backend reliability pilot

One successful paired record establishes basic technical feasibility but is
insufficient to authorise 400 batch requests. Before full inference, a
five-record reliability pilot tests whether the OpenCode-backed route continues
to return valid responses across additional network-flow records.

The reliability sample is selected deterministically and without loading ground
truth. Five positions are distributed across the existing 200-record evaluation
order: the first, quarter, midpoint, three-quarter and final positions. This is
an operational coverage strategy rather than a statistical sample for estimating
detection performance.

`pilot_001` has already completed successfully under the locked OpenCode prompt
template and is retained as the first reliability pair. Four additional records
therefore require eight new requests.

No model, prompt-template, schema, feature-set or backend setting is changed
after observing the first valid pair. Reliability-pilot predictions will not be
compared with ground truth at this stage.

The following cell selects the records, constructs their paired prompts and
audits the request plan. It makes no external request.

In [24]:
RELIABILITY_PILOT_RECORD_COUNT = 5

# Select positions across the complete evaluation order without consulting
# labels or any other private ground-truth information.
reliability_pilot_indices = [
    0,
    len(structured_records) // 4,
    len(structured_records) // 2,
    3 * len(structured_records) // 4,
    len(structured_records) - 1,
]

assert reliability_pilot_indices == [
    0,
    50,
    100,
    150,
    199,
]
assert (
    len(reliability_pilot_indices)
    == RELIABILITY_PILOT_RECORD_COUNT
)
assert len(set(reliability_pilot_indices)) == 5

reliability_pilot_request_plan = []

for record_index in reliability_pilot_indices:
    structured_record = structured_records[record_index]
    text_record = text_records[record_index]

    # Confirm that the two representations belong to the same underlying
    # record before constructing either provider-specific prompt.
    assert (
        structured_record["sample_id"]
        == text_record["sample_id"]
    )
    assert (
        structured_record["canonical_payload_sha256"]
        == text_record["canonical_payload_sha256"]
    )
    assert (
        structured_record["feature_set_id"]
        == text_record["feature_set_id"]
        == "primary_46"
    )

    sample_id = structured_record["sample_id"]

    structured_contract = build_logical_request_contract(
        structured_record["model_input"]
    )
    text_contract = build_logical_request_contract(
        text_record["model_input"]
    )

    structured_prompt = build_opencode_research_prompt(
        structured_contract
    )
    text_prompt = build_opencode_research_prompt(
        text_contract
    )

    structured_prompt_sha256 = stable_json_sha256(
        {
            "template_version": (
                OPENCODE_PROMPT_TEMPLATE_VERSION
            ),
            "prompt": structured_prompt,
        }
    )

    text_prompt_sha256 = stable_json_sha256(
        {
            "template_version": (
                OPENCODE_PROMPT_TEMPLATE_VERSION
            ),
            "prompt": text_prompt,
        }
    )

    grounding_reference = build_grounding_reference(
        structured_record["model_input"]
    )

    assert len(grounding_reference) == 46
    assert sample_id not in structured_prompt
    assert sample_id not in text_prompt
    assert structured_prompt != text_prompt

    reliability_pilot_request_plan.append(
        {
            "record_index": record_index,
            "sample_id": sample_id,
            "condition": "structured",
            "canonical_payload_sha256": (
                structured_record[
                    "canonical_payload_sha256"
                ]
            ),
            "prompt": structured_prompt,
            "prompt_sha256": (
                structured_prompt_sha256
            ),
            "grounding_reference": grounding_reference,
            "already_completed_as_preflight": (
                sample_id == preflight_sample_id
            ),
        }
    )

    reliability_pilot_request_plan.append(
        {
            "record_index": record_index,
            "sample_id": sample_id,
            "condition": "deterministic_text",
            "canonical_payload_sha256": (
                text_record[
                    "canonical_payload_sha256"
                ]
            ),
            "prompt": text_prompt,
            "prompt_sha256": text_prompt_sha256,
            "grounding_reference": grounding_reference,
            "already_completed_as_preflight": (
                sample_id == preflight_sample_id
            ),
        }
    )


# The plan contains ten condition-level entries: two representations for each
# of five records. The first pair is already complete, leaving eight new calls.
assert len(reliability_pilot_request_plan) == 10

new_reliability_requests = [
    request
    for request in reliability_pilot_request_plan
    if not request["already_completed_as_preflight"]
]

assert len(new_reliability_requests) == 8

reliability_pilot_plan_summary = pd.DataFrame(
    [
        {
            "record_index": request["record_index"],
            "sample_id": request["sample_id"],
            "condition": request["condition"],
            "prompt_characters": len(
                request["prompt"]
            ),
            "prompt_sha256": (
                request["prompt_sha256"]
            ),
            "grounding_reference_feature_count": len(
                request["grounding_reference"]
            ),
            "already_completed_as_preflight": (
                request[
                    "already_completed_as_preflight"
                ]
            ),
            "requires_new_request": (
                not request[
                    "already_completed_as_preflight"
                ]
            ),
        }
        for request in reliability_pilot_request_plan
    ]
)

display(reliability_pilot_plan_summary)

pd.Series(
    {
        "selected_record_indices": (
            reliability_pilot_indices
        ),
        "selected_record_count": (
            RELIABILITY_PILOT_RECORD_COUNT
        ),
        "planned_condition_level_results": len(
            reliability_pilot_request_plan
        ),
        "results_already_completed": (
            len(reliability_pilot_request_plan)
            - len(new_reliability_requests)
        ),
        "new_external_requests_required": len(
            new_reliability_requests
        ),
        "all_grounding_references_have_46_features": all(
            len(request["grounding_reference"]) == 46
            for request in reliability_pilot_request_plan
        ),
        "ground_truth_loaded": False,
        "network_request_made": False,
    },
    name="value",
)

,record_index,sample_id,condition,prompt_characters,prompt_sha256,grounding_reference_feature_count,already_completed_as_preflight,requires_new_request
0,0,pilot_001,structured,4049,f49c1346affe94d3a4be85086cfc5f362c91bcbbfb992d...,46,True,False
1,0,pilot_001,deterministic_text,4026,577d3214b51eed9805ffb1291cc5643c6711e78b0f2718...,46,True,False
2,50,pilot_051,structured,4056,b4ecc3304518c130130b92341465137a089fd2b1d8115a...,46,False,True
3,50,pilot_051,deterministic_text,4033,6554e3f088749e72ec91843c2e846cecf7a2c67513eb41...,46,False,True
4,100,pilot_101,structured,4046,9c4e79f99a2b8893c1268dedda087edaeb585bb5104245...,46,False,True
5,100,pilot_101,deterministic_text,4023,b729468c073ce59efcc289c3d93a1b166a69bb726defe8...,46,False,True
6,150,pilot_151,structured,4051,4e2f8d239643863604d45f388ab0edbffbbe454bfa8f1e...,46,False,True
7,150,pilot_151,deterministic_text,4028,d721d831bf6995f99cc98949ed5021261b9ad7e39d8b0f...,46,False,True
8,199,pilot_200,structured,4052,01afa457a334e4f18fdd8aca8d7d6de79e5a9964aeee26...,46,False,True
9,199,pilot_200,deterministic_text,4029,ee9037a1c143d3275b5ed76a9604c3a01e9373efa0734e...,46,False,True


selected_record_indices                      [0, 50, 100, 150, 199]
selected_record_count                                             5
planned_condition_level_results                                  10
results_already_completed                                         2
new_external_requests_required                                    8
all_grounding_references_have_46_features                      True
ground_truth_loaded                                           False
network_request_made                                          False
Name: value, dtype: object

### Reliability-pilot checkpoint strategy

Each condition-level result is persisted as a separate atomic JSON checkpoint
immediately after its request finishes. This design avoids holding all new
responses only in kernel memory and prevents a later interruption from
invalidating earlier completed work.

Checkpoint identity is determined by the original evaluation-record index and
representation condition. Before skipping an existing checkpoint, the runner
verifies its sample identifier, condition, canonical-payload hash and prompt
hash against the current locked request plan. A filename alone is therefore
insufficient to authorise a skip.

The two valid `pilot_001` preflight responses are copied into the reliability
checkpoint set without making new requests. The remaining eight requests are
executed sequentially. Sequential execution is retained for this operational
pilot so that concurrency is not introduced before backend reliability has been
established.

Each checkpoint excludes ground truth and hidden reasoning content. Failed,
timed-out or invalid responses are also checkpointed rather than silently
retried, because they are part of the backend-reliability evidence.

In [25]:
RELIABILITY_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "reliability"
    / "primary_46"
)

RELIABILITY_CHECKPOINT_DIR = (
    RELIABILITY_RESULTS_DIR / "checkpoints"
)


def reliability_checkpoint_path(
    record_index: int,
    condition: str,
) -> Path:
    """
    Return the unique checkpoint path for one record-condition request.
    """
    if condition not in ALLOWED_CONDITIONS:
        raise ValueError(
            f"Unsupported condition: {condition!r}."
        )

    if not isinstance(record_index, int) or record_index < 0:
        raise ValueError(
            "record_index must be a non-negative integer."
        )

    return (
        RELIABILITY_CHECKPOINT_DIR
        / f"record_{record_index:04d}__{condition}.json"
    )


def read_reliability_checkpoint(
    path: Path,
) -> dict:
    """Read and validate the top-level type of one checkpoint."""
    with path.open("r", encoding="utf-8") as handle:
        checkpoint = json.load(handle)

    if not isinstance(checkpoint, dict):
        raise ValueError(
            f"Checkpoint is not a JSON object: {path}"
        )

    return checkpoint


def verify_checkpoint_matches_plan(
    checkpoint: dict,
    planned_request: dict,
) -> None:
    """
    Verify that an existing checkpoint belongs to the current locked request.

    This prevents an old response from being skipped merely because its
    filename happens to match the current record and condition.
    """
    expected_fields = {
        "record_index": planned_request["record_index"],
        "sample_id": planned_request["sample_id"],
        "condition": planned_request["condition"],
        "canonical_payload_sha256": (
            planned_request["canonical_payload_sha256"]
        ),
        "prompt_sha256": planned_request["prompt_sha256"],
        "feature_set_id": "primary_46",
        "backend": "opencode",
        "opencode_version": OPENCODE_VERSION,
        "requested_model": OPENCODE_UOA_MODEL,
        "prompt_template_version": (
            OPENCODE_PROMPT_TEMPLATE_VERSION
        ),
    }

    mismatches = {
        field_name: {
            "expected": expected_value,
            "observed": checkpoint.get(field_name),
        }
        for field_name, expected_value in expected_fields.items()
        if checkpoint.get(field_name) != expected_value
    }

    if mismatches:
        raise ValueError(
            "Existing checkpoint does not match the current "
            f"request plan: {mismatches}"
        )


def persist_reliability_checkpoint(
    result: dict,
    planned_request: dict,
    checkpoint_origin: str,
) -> str:
    """
    Persist one result atomically or verify an already existing checkpoint.

    Returns either `written` or `verified_existing`.
    """
    checkpoint_path = reliability_checkpoint_path(
        planned_request["record_index"],
        planned_request["condition"],
    )

    if checkpoint_path.exists():
        existing_checkpoint = read_reliability_checkpoint(
            checkpoint_path
        )
        verify_checkpoint_matches_plan(
            existing_checkpoint,
            planned_request,
        )
        return "verified_existing"

    checkpoint_record = {
        **result,
        "record_index": planned_request["record_index"],
        "experiment_phase": "backend_reliability_pilot",
        "checkpoint_origin": checkpoint_origin,
        "checkpoint_created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    verify_checkpoint_matches_plan(
        checkpoint_record,
        planned_request,
    )

    write_new_text_file(
        checkpoint_path,
        json.dumps(
            checkpoint_record,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        )
        + "\n",
    )

    # Read the file back immediately so persistence and JSON decoding are
    # verified before execution can continue to another request.
    persisted_checkpoint = read_reliability_checkpoint(
        checkpoint_path
    )
    verify_checkpoint_matches_plan(
        persisted_checkpoint,
        planned_request,
    )

    return "written"


RELIABILITY_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Register the two already completed pilot_001 results as the first reliability
# pair. No request is repeated.
preflight_plan_entries = [
    request
    for request in reliability_pilot_request_plan
    if request["already_completed_as_preflight"]
]

assert len(preflight_plan_entries) == 2

preflight_results_by_condition = {
    result["condition"]: result
    for result in opencode_paired_preflight_results
}

preflight_checkpoint_actions = []

for planned_request in preflight_plan_entries:
    matching_preflight_result = (
        preflight_results_by_condition[
            planned_request["condition"]
        ]
    )

    action = persist_reliability_checkpoint(
        result=matching_preflight_result,
        planned_request=planned_request,
        checkpoint_origin="paired_preflight",
    )

    preflight_checkpoint_actions.append(
        {
            "record_index": planned_request["record_index"],
            "sample_id": planned_request["sample_id"],
            "condition": planned_request["condition"],
            "checkpoint_action": action,
            "network_request_made": False,
        }
    )

display(pd.DataFrame(preflight_checkpoint_actions))

pd.Series(
    {
        "checkpoint_directory": (
            RELIABILITY_CHECKPOINT_DIR
            .relative_to(PROJECT_ROOT)
            .as_posix()
        ),
        "preflight_checkpoints_expected": 2,
        "preflight_checkpoints_present": sum(
            reliability_checkpoint_path(
                request["record_index"],
                request["condition"],
            ).exists()
            for request in preflight_plan_entries
        ),
        "new_external_requests_made": 0,
        "ground_truth_loaded": False,
        "existing_valid_checkpoints_overwritten": False,
    },
    name="value",
)

,record_index,sample_id,condition,checkpoint_action,network_request_made
0,0,pilot_001,structured,written,False
1,0,pilot_001,deterministic_text,written,False


checkpoint_directory                      results/reliability/primary_46/checkpoints
preflight_checkpoints_expected                                                     2
preflight_checkpoints_present                                                      2
new_external_requests_made                                                         0
ground_truth_loaded                                                            False
existing_valid_checkpoints_overwritten                                         False
Name: value, dtype: object

In [26]:
reliability_execution_progress = []

for request_number, planned_request in enumerate(
    new_reliability_requests,
    start=1,
):
    checkpoint_path = reliability_checkpoint_path(
        planned_request["record_index"],
        planned_request["condition"],
    )

    # Resume safely: an existing file is skipped only after its identifiers
    # and hashes have been verified against the current request plan.
    if checkpoint_path.exists():
        existing_checkpoint = read_reliability_checkpoint(
            checkpoint_path
        )
        verify_checkpoint_matches_plan(
            existing_checkpoint,
            planned_request,
        )

        reliability_execution_progress.append(
            {
                "request_number": request_number,
                "sample_id": planned_request["sample_id"],
                "condition": planned_request["condition"],
                "action": "verified_existing_and_skipped",
                "request_completed": existing_checkpoint.get(
                    "request_completed"
                ),
                "overall_response_valid": (
                    existing_checkpoint.get(
                        "overall_response_valid"
                    )
                ),
                "elapsed_seconds": existing_checkpoint.get(
                    "elapsed_seconds"
                ),
            }
        )
        continue

    result = run_opencode_research_request(
        prompt=planned_request["prompt"],
        prompt_sha256=(
            planned_request["prompt_sha256"]
        ),
        condition=planned_request["condition"],
        local_sample_id=planned_request["sample_id"],
        canonical_payload_sha256=(
            planned_request["canonical_payload_sha256"]
        ),
        grounding_reference=(
            planned_request["grounding_reference"]
        ),
    )

    checkpoint_action = persist_reliability_checkpoint(
        result=result,
        planned_request=planned_request,
        checkpoint_origin="reliability_execution",
    )

    reliability_execution_progress.append(
        {
            "request_number": request_number,
            "sample_id": planned_request["sample_id"],
            "condition": planned_request["condition"],
            "action": checkpoint_action,
            "request_completed": result[
                "request_completed"
            ],
            "overall_response_valid": result[
                "overall_response_valid"
            ],
            "elapsed_seconds": result[
                "elapsed_seconds"
            ],
        }
    )

    # Stop rather than repeatedly contacting a backend that reports exhausted
    # quota. Other invalid responses remain checkpointed and do not trigger an
    # automatic retry.
    if result["budget_exhausted"]:
        break


display(pd.DataFrame(reliability_execution_progress))

,request_number,sample_id,condition,action,request_completed,overall_response_valid,elapsed_seconds
0,1,pilot_051,structured,written,True,True,48.059807
1,2,pilot_051,deterministic_text,written,True,True,69.980798
2,3,pilot_101,structured,written,True,True,31.405777
3,4,pilot_101,deterministic_text,written,True,True,19.748620
4,5,pilot_151,structured,written,True,True,16.739961
5,6,pilot_151,deterministic_text,written,True,True,17.313443
6,7,pilot_200,structured,written,True,True,26.124701
7,8,pilot_200,deterministic_text,written,True,True,73.606603


In [27]:
reliability_checkpoint_records = []

for planned_request in reliability_pilot_request_plan:
    checkpoint_path = reliability_checkpoint_path(
        planned_request["record_index"],
        planned_request["condition"],
    )

    if not checkpoint_path.exists():
        continue

    checkpoint = read_reliability_checkpoint(
        checkpoint_path
    )

    verify_checkpoint_matches_plan(
        checkpoint,
        planned_request,
    )

    reliability_checkpoint_records.append(checkpoint)


reliability_summary_columns = [
    "record_index",
    "sample_id",
    "condition",
    "checkpoint_origin",
    "request_completed",
    "return_code",
    "elapsed_seconds",
    "finish_reasons",
    "input_tokens",
    "output_tokens",
    "reasoning_tokens",
    "total_tokens",
    "visible_json_parsed",
    "schema_structure_valid",
    "grounding_valid",
    "overall_response_valid",
    "budget_exhausted",
    "error_type",
]

reliability_checkpoint_summary = pd.DataFrame(
    reliability_checkpoint_records
)[reliability_summary_columns].sort_values(
    ["record_index", "condition"]
)

display(reliability_checkpoint_summary)

pd.Series(
    {
        "expected_checkpoint_count": 10,
        "observed_checkpoint_count": len(
            reliability_checkpoint_records
        ),
        "completed_request_count": sum(
            bool(record["request_completed"])
            for record in reliability_checkpoint_records
        ),
        "overall_valid_response_count": sum(
            bool(record["overall_response_valid"])
            for record in reliability_checkpoint_records
        ),
        "invalid_response_count": sum(
            not bool(record["overall_response_valid"])
            for record in reliability_checkpoint_records
        ),
        "budget_exhaustion_observed": any(
            bool(record["budget_exhausted"])
            for record in reliability_checkpoint_records
        ),
        "ground_truth_loaded": False,
        "network_request_made_by_this_summary_cell": False,
    },
    name="value",
)

,record_index,sample_id,condition,checkpoint_origin,request_completed,return_code,elapsed_seconds,finish_reasons,input_tokens,output_tokens,reasoning_tokens,total_tokens,visible_json_parsed,schema_structure_valid,grounding_valid,overall_response_valid,budget_exhausted,error_type
1,0,pilot_001,deterministic_text,paired_preflight,True,0,24.221286,[stop],8609,428,1471,10508,True,True,True,True,False,None
0,0,pilot_001,structured,paired_preflight,True,0,21.449245,[stop],8664,368,1043,10075,True,True,True,True,False,None
3,50,pilot_051,deterministic_text,reliability_execution,True,0,69.980798,[stop],8613,838,5494,14945,True,True,True,True,False,None
2,50,pilot_051,structured,reliability_execution,True,0,48.059807,[stop],8671,784,3274,12729,True,True,True,True,False,None
5,100,pilot_101,deterministic_text,reliability_execution,True,0,19.748620,[stop],8605,414,1023,10042,True,True,True,True,False,None
4,100,pilot_101,structured,reliability_execution,True,0,31.405777,[stop],8658,597,1975,11230,True,True,True,True,False,None
7,150,pilot_151,deterministic_text,reliability_execution,True,0,17.313443,[stop],8612,467,744,9823,True,True,True,True,False,None
6,150,pilot_151,structured,reliability_execution,True,0,16.739961,[stop],8664,341,808,9813,True,True,True,True,False,None
9,199,pilot_200,deterministic_text,reliability_execution,True,0,73.606603,[stop],8612,905,5776,15293,True,True,True,True,False,None
8,199,pilot_200,structured,reliability_execution,True,0,26.124701,[stop],8665,487,1587,10739,True,True,True,True,False,None


expected_checkpoint_count                       10
observed_checkpoint_count                       10
completed_request_count                         10
overall_valid_response_count                    10
invalid_response_count                           0
budget_exhaustion_observed                   False
ground_truth_loaded                          False
network_request_made_by_this_summary_cell    False
Name: value, dtype: object

### Reliability-pilot gate decision

The five-record reliability pilot produced all ten expected condition-level
checkpoints. Every request completed, every visible response passed schema and
grounding validation, no budget-exhaustion marker was observed, and no
ground-truth data was loaded.

The observed request duration varied substantially across records and
representations, from approximately 16.74 to 73.61 seconds. This supports
retaining the 300-second per-request timeout and confirms that short probe
latency should not be treated as representative of research-request latency.

The OpenCode-backed route is therefore authorised for resumable batch
inference under the same model route, prompt-template version, feature set,
validation rules and safety controls.

This decision concerns backend reliability only. The reliability-pilot
predictions are not evaluated for correctness at this stage.

The following cell aggregates the ten immutable checkpoints into one JSON Lines
artifact and creates a reliability manifest. It makes no external request and
does not overwrite existing artifacts.

In [28]:
RELIABILITY_RESULTS_PATH = (
    RELIABILITY_RESULTS_DIR
    / "opencode_reliability_results.jsonl"
)

RELIABILITY_MANIFEST_PATH = (
    RELIABILITY_RESULTS_DIR
    / "opencode_reliability_manifest.json"
)

# Sort deterministically by the original evaluation order and representation
# order. This makes the aggregate file reproducible regardless of the order in
# which individual checkpoints were completed.
condition_sort_order = {
    "structured": 0,
    "deterministic_text": 1,
}

sorted_reliability_records = sorted(
    reliability_checkpoint_records,
    key=lambda record: (
        record["record_index"],
        condition_sort_order[record["condition"]],
    ),
)

assert len(sorted_reliability_records) == 10
assert all(
    record["request_completed"]
    for record in sorted_reliability_records
)
assert all(
    record["overall_response_valid"]
    for record in sorted_reliability_records
)
assert all(
    not record["ground_truth_loaded"]
    for record in sorted_reliability_records
)

reliability_results_jsonl = (
    "\n".join(
        json.dumps(
            record,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        for record in sorted_reliability_records
    )
    + "\n"
)

write_new_text_file(
    RELIABILITY_RESULTS_PATH,
    reliability_results_jsonl,
)

reliability_results_sha256 = sha256_file(
    RELIABILITY_RESULTS_PATH
)

reliability_elapsed_seconds = [
    float(record["elapsed_seconds"])
    for record in sorted_reliability_records
]

reliability_total_tokens = [
    int(record["total_tokens"])
    for record in sorted_reliability_records
]

reliability_manifest = {
    "artifact_type": "backend_reliability_manifest",
    "artifact_version": "0.1.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "experiment_scope": "five_record_backend_reliability_pilot",
    "selection": {
        "ground_truth_blind": True,
        "record_indices": reliability_pilot_indices,
        "record_count": RELIABILITY_PILOT_RECORD_COUNT,
        "condition_level_result_count": len(
            sorted_reliability_records
        ),
    },
    "feature_set": {
        "id": "primary_46",
        "feature_count": 46,
    },
    "backend": {
        "name": "opencode",
        "version": OPENCODE_VERSION,
        "model_route": OPENCODE_UOA_MODEL,
        "prompt_template_version": (
            OPENCODE_PROMPT_TEMPLATE_VERSION
        ),
        "pure_mode": True,
        "empty_temporary_directory": True,
        "auto_approval": False,
        "files_attached": False,
        "subprocess_timeout_seconds": (
            OPENCODE_REQUEST_TIMEOUT_SECONDS
        ),
        "maximum_model_output_tokens_supplied": False,
    },
    "validation": {
        "expected_checkpoint_count": 10,
        "observed_checkpoint_count": len(
            sorted_reliability_records
        ),
        "completed_request_count": sum(
            bool(record["request_completed"])
            for record in sorted_reliability_records
        ),
        "schema_valid_count": sum(
            bool(record["schema_structure_valid"])
            for record in sorted_reliability_records
        ),
        "grounding_valid_count": sum(
            bool(record["grounding_valid"])
            for record in sorted_reliability_records
        ),
        "overall_valid_count": sum(
            bool(record["overall_response_valid"])
            for record in sorted_reliability_records
        ),
        "invalid_count": sum(
            not bool(record["overall_response_valid"])
            for record in sorted_reliability_records
        ),
        "budget_exhaustion_observed": any(
            bool(record["budget_exhausted"])
            for record in sorted_reliability_records
        ),
        "ground_truth_loaded": False,
        "batch_inference_authorised": all(
            bool(record["overall_response_valid"])
            for record in sorted_reliability_records
        ),
    },
    "operational_summary": {
        "minimum_elapsed_seconds": min(
            reliability_elapsed_seconds
        ),
        "maximum_elapsed_seconds": max(
            reliability_elapsed_seconds
        ),
        "mean_elapsed_seconds": (
            sum(reliability_elapsed_seconds)
            / len(reliability_elapsed_seconds)
        ),
        "minimum_total_tokens": min(
            reliability_total_tokens
        ),
        "maximum_total_tokens": max(
            reliability_total_tokens
        ),
        "mean_total_tokens": (
            sum(reliability_total_tokens)
            / len(reliability_total_tokens)
        ),
    },
    "protocol": {
        "path": PROTOCOL_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "sha256": sha256_file(PROTOCOL_PATH),
    },
    "output_schema": {
        "path": OUTPUT_SCHEMA_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "sha256": sha256_file(
            OUTPUT_SCHEMA_PATH
        ),
    },
    "result_file": {
        "path": RELIABILITY_RESULTS_PATH.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "sha256": reliability_results_sha256,
    },
}

assert (
    reliability_manifest["validation"][
        "batch_inference_authorised"
    ]
    is True
)

write_new_text_file(
    RELIABILITY_MANIFEST_PATH,
    json.dumps(
        reliability_manifest,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )
    + "\n",
)

reliability_persistence_summary = pd.Series(
    {
        "results_file": (
            RELIABILITY_RESULTS_PATH
            .relative_to(PROJECT_ROOT)
            .as_posix()
        ),
        "manifest_file": (
            RELIABILITY_MANIFEST_PATH
            .relative_to(PROJECT_ROOT)
            .as_posix()
        ),
        "persisted_result_count": len(
            sorted_reliability_records
        ),
        "results_file_sha256": (
            reliability_results_sha256
        ),
        "manifest_file_sha256": sha256_file(
            RELIABILITY_MANIFEST_PATH
        ),
        "completed_request_count": (
            reliability_manifest["validation"][
                "completed_request_count"
            ]
        ),
        "overall_valid_count": (
            reliability_manifest["validation"][
                "overall_valid_count"
            ]
        ),
        "batch_inference_authorised": (
            reliability_manifest["validation"][
                "batch_inference_authorised"
            ]
        ),
        "ground_truth_saved": False,
        "reasoning_content_saved": False,
        "network_request_made": False,
    },
    name="value",
)

reliability_persistence_summary

results_file                  results/reliability/primary_46/opencode_reliab...
manifest_file                 results/reliability/primary_46/opencode_reliab...
persisted_result_count                                                       10
results_file_sha256           90bdf0594e33a9a6c86a8b13096ac147454ef7bd7999a2...
manifest_file_sha256          a3af39ed2123b22a8450bfd29b85de984d27ec8a16cd88...
completed_request_count                                                      10
overall_valid_count                                                          10
batch_inference_authorised                                                 True
ground_truth_saved                                                        False
reasoning_content_saved                                                   False
network_request_made                                                      False
Name: value, dtype: object

## Notebook conclusion and handoff

### Purpose

This notebook evaluated candidate execution paths for the controlled
LLM-based network-flow anomaly-detection experiment before authorising full
batch inference.

The evaluated task used:

* the `primary_46` feature set;
* 200 paired network-flow records;
* structured JSON and deterministic natural-language representations;
* the binary labels `Benign` and `DoS`;
* one common research instruction and response schema; and
* local schema and feature-grounding validation without loading ground truth.

The objective of this notebook was operational validation rather than
predictive-performance evaluation.

### Direct API findings

The UoA-hosted `MiniMax-M3` model was first evaluated through the
OpenAI-compatible Chat Completions API.

The gateway accepted the required model, temperature, seed and structured
response parameters. However, provider reasoning continued to be generated
even when a thinking-disable parameter was accepted. Full research requests
exhausted the available completion allowance before returning valid visible
responses.

The same model was subsequently tested through the Responses API. A minimal
non-research structured-output probe completed successfully, confirming
endpoint compatibility. Nevertheless, both full paired research requests
reached the configured 8,192-token output limit and returned `incomplete`
status. Neither visible response was complete JSON.

These results establish that successful connectivity and parameter acceptance
were insufficient to support the full research task through the tested direct
API paths. The failed attempts are retained as backend-feasibility evidence and
are not treated as anomaly-detection observations.

### OpenCode backend validation

A locally configured OpenCode 1.18.22 route to `uoa/MiniMax-M3` was then tested
as an alternative execution backend.

The OpenCode configuration used:

* `--pure`;
* a newly created empty temporary directory for each request;
* no file attachments;
* no automatic permission approval;
* JSON event output;
* a 300-second subprocess timeout; and
* no notebook-supplied model-output token limit.

The minimal OpenCode probe completed successfully and returned the expected
visible response. Event inspection showed that OpenCode adds a substantial
internal agent context even under `--pure` mode. OpenCode is therefore treated
as a distinct execution backend rather than a request-equivalent replacement
for the direct API.

Because the CLI accepts a single prompt rather than separate system and user
messages, the locked research instruction, JSON Schema and
representation-specific record were combined using deterministic prompt
template version `0.1.0`. The non-record portions of the structured and
deterministic-text prompts were verified to be identical.

### Paired preflight outcome

The first OpenCode-backed paired research preflight completed successfully for
both representations of `pilot_001`.

Both responses:

* returned exit code zero;
* ended with reason `stop`;
* parsed as JSON;
* satisfied the required response structure;
* cited exactly five distinct supplied features;
* copied the cited feature names and observed values correctly; and
* passed overall local validation.

Both representations predicted `Benign` for this record. Three of the five
cited features were shared, producing a top-five Jaccard similarity of
approximately `0.4286`, while no feature occupied the same rank in both
responses.

This is a single-record operational observation, not a research conclusion. It
nevertheless demonstrates why classification agreement and explanation
agreement must be evaluated separately.

### Reliability-pilot outcome

A ground-truth-blind reliability pilot selected five records from positions
`0`, `50`, `100`, `150` and `199` of the existing evaluation order. Each record
was evaluated in both representation conditions, producing ten
condition-level results.

All ten requests:

* completed within the 300-second timeout;
* produced valid schema-conforming JSON;
* passed exact feature-name and observed-value grounding checks;
* avoided budget-exhaustion indicators; and
* were persisted as individual atomic checkpoints.

Observed request latency ranged from approximately 16.74 to 73.61 seconds.
This variation confirms the need for a generous bounded timeout and resumable
checkpointing during full inference.

The reliability gate passed with:

* 10 expected checkpoints;
* 10 completed requests;
* 10 overall-valid responses; and
* 0 invalid responses.

The OpenCode-backed route is therefore technically authorised for resumable
batch inference.

### Security, privacy and interpretation boundaries

No API key or provider credential is stored in this notebook or its result
artifacts. Hidden provider reasoning text is not retained. Ground truth remains
unloaded and is not included in any prompt, response checkpoint or
backend-validation artifact.

OpenCode is not an operating-system sandbox. The empty-directory, `--pure`,
no-attachment and no-auto-approval controls reduce incidental context and tool
risk but do not establish complete process isolation. This limitation must be
reported in the final methodology.

The results in this notebook establish backend feasibility and reliability
only. They do not establish anomaly-detection accuracy, representation effects
or explanation correctness.

### Handoff to formal batch inference

Formal inference will be implemented in
`07_llm_batch_inference.ipynb`.

Notebook 07 will:

1. construct the complete 200-record paired request plan;
2. retain the locked 46-feature set and OpenCode prompt template;
3. reuse compatible validated checkpoints without repeating requests;
4. balance representation execution order where possible;
5. limit OpenCode concurrency to a documented maximum;
6. persist each condition-level result atomically;
7. support interruption-safe resume;
8. stop on detected budget exhaustion;
9. verify that every sample has exactly one result per representation; and
10. keep ground truth inaccessible throughout inference.

Ground truth will first be joined in a later evaluation notebook after the
complete inference artifact has been frozen.
